# 🏋️ 운동 보조 시스템 v5 — ST-GCN 모델 종목/횟수 인식 버전
**캡스톤 디자인 | 스쿼트 기준 | 원본비율 유지 + 전문가 skeleton + 몸통 zone + 안정화 HUD**

---
### ✅ v5에서 달라진 점
1. **피드백 문장 개선**: `값이 전문가보다 큽니다/작습니다` 대신 `몸통 기울기 과도`, `무릎이 덜 굽혀짐`처럼 의미 중심 문장으로 표시합니다.
2. **불안정 지표 제외**: 측면영상에서 흔들리기 쉬운 `neck_lean`, `foot_width`, `hand_width` 계열은 핵심 오류판정에서 제외합니다.
3. **피드백 안정화**: 같은 오류가 3프레임 정도 연속될 때만 핵심 피드백을 바꾸어 깜빡임을 줄입니다.
4. **HUD 개선**: 현재 구간, `U=사용자 / E=전문가 / Δ=차이` 설명, 핵심 피드백 카드를 크게 표시합니다.
5. **빨간 원 최소화**: 현재 우선 확인해야 하는 관절만 표시해 화면이 덜 복잡하게 보이도록 했습니다.

---
### 실행 순서
Cell 1부터 Cell 10까지 순서대로 실행하면 됩니다. Cell 7.5가 v5 핵심 업그레이드 셀입니다.


> 이 버전은 기존 v5 피드백/시각화 흐름은 유지하고, 종목 인식과 횟수 카운트만 `model/realtime_stgcn_infer.py`의 학습 ST-GCN 모델 출력으로 교체합니다.


## 실행 순서
1. **경로 입력 셀(Cell 3A)**에서 `USER_VIDEO_PATH`, `EXPERT_JSON_PATH`, `POSE_MODEL_PATH`, `STGCN_CHECKPOINT_PATH`를 수정합니다.
2. **모델/실행 옵션 셀(Cell 3B)**에서 `EXERCISE`, `VIDEO_VIEW_MODE`, 모델 사용 옵션을 수정합니다.
3. 위에서 아래로 한 셀씩 실행합니다.
4. 마지막 메인 루프 셀은 출력 영상을 생성합니다.
5. 메인 루프가 끝나면 `landmarker.close()`가 호출됩니다. 같은 커널에서 다시 처리하려면 **Cell 8A → 9C → 9D → 10** 순서로 다시 실행하세요.

6. 이 버전은 `realtime_compare_side.py`, `feedback_overlay.py`, `model/realtime_stgcn_infer.py`를 import하지 않고 필요한 코드를 노트북 셀 안에 포함합니다.


In [1]:
# ── Cell 2: 버전 확인 ─────────────────────────────────────────────
import numpy as np
import cv2
import mediapipe as mp
from PIL import Image

print('numpy    :', np.__version__)
print('cv2      :', cv2.__version__)
print('mediapipe:', mp.__version__)
print('Pillow   :', Image.__version__)

numpy    : 2.4.4
cv2      : 4.13.0
mediapipe: 0.10.14
Pillow   : 12.2.0


In [2]:
# ── Cell 3A: 사용자가 수정하는 경로 입력만 모아둔 셀 ─────────────
import os
import glob

# 로컬 프로젝트 경로
BASE = r'C:\Users\kojy1\PycharmProjects\capstone'

# 1) 입력 영상 경로
USER_VIDEO_PATH = os.path.join(BASE, 'data',  'merged_dead_squat.mp4')
# 다른 예시:
# USER_VIDEO_PATH = os.path.join(BASE, 'data', 'squat_user2.mp4')
# USER_VIDEO_PATH = os.path.join(BASE, 'data', '측면', 'data', 'squat', 'squat_15.mp4')

# 2) 전문가 JSON 경로
EXPERT_JSON_PATH = os.path.join(BASE, 'expert_dead2.json')
# squat 전문가 JSON이 있으면 예:
# EXPERT_JSON_PATH = os.path.join(BASE, 'expert_squat.json')

# 3) MediaPipe PoseLandmarker task 모델 경로
POSE_MODEL_PATH = os.path.join(BASE, 'pose_landmarker_lite.task')

# 4) ST-GCN 학습 checkpoint 경로
STGCN_CHECKPOINT_PATH = os.path.join(
    BASE,
    'phase_experiments',
    'temporal_head_full_ablation',
    'mlp_h128_c16_pw_2.0_ts2_do03_aug1',
    'best_ep013.pt',
)
# None으로 바꾸면 model/realtime_stgcn_infer.py가 기본 best/latest checkpoint를 자동 탐색합니다.
# STGCN_CHECKPOINT_PATH = None

# 5) 출력 영상 경로
OUTPUT_DIR = os.path.join(BASE, 'output')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'dead_v5_trunk_foot_upgrade_model_infer.mp4')

# 6) 한글 폰트 경로
FONT_PATH = r'C:\Windows\Fonts\malgun.ttf'

print('경로 입력 완료')
print(f'BASE              = {BASE}')
print(f'USER_VIDEO_PATH   = {USER_VIDEO_PATH}')
print(f'EXPERT_JSON_PATH  = {EXPERT_JSON_PATH}')
print(f'POSE_MODEL_PATH   = {POSE_MODEL_PATH}')
print(f'STGCN_CHECKPOINT  = {STGCN_CHECKPOINT_PATH}')
print(f'OUTPUT_PATH       = {OUTPUT_PATH}')


경로 입력 완료
BASE              = C:\Users\kojy1\PycharmProjects\capstone
USER_VIDEO_PATH   = C:\Users\kojy1\PycharmProjects\capstone\data\merged_dead_squat.mp4
EXPERT_JSON_PATH  = C:\Users\kojy1\PycharmProjects\capstone\expert_dead2.json
POSE_MODEL_PATH   = C:\Users\kojy1\PycharmProjects\capstone\pose_landmarker_lite.task
STGCN_CHECKPOINT  = C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\temporal_head_full_ablation\mlp_h128_c16_pw_2.0_ts2_do03_aug1\best_ep013.pt
OUTPUT_PATH       = C:\Users\kojy1\PycharmProjects\capstone\output\dead_v5_trunk_foot_upgrade_model_infer.mp4


In [3]:
# ── Cell 3B: 모델/실행 옵션만 모아둔 셀 ────────────────────────
# 전문가 JSON/피드백 기준 운동입니다.
# 현재 로컬 경로는 deadlift 영상/전문가 JSON에 맞춰져 있습니다.
# squat 영상+expert_squat.json을 쓸 때는 EXERCISE = 'squat'으로 바꾸세요.
EXERCISE = '모델 작동안하는중'

# 촬영 방향 설정
# - 완전 측면: 'side'
# - 정면: 'front'
# - 45도 사선: '45deg'
# - 자동 추정: 'auto'
VIDEO_VIEW_MODE = 'side'

# ST-GCN 모델 사용 옵션
MODEL_DRIVEN_RECOGNITION = True
MODEL_DRIVEN_COUNT = True
TRAINED_MODEL_REQUIRED = True       # True: 모델 실패 시 에러발생
TRAINED_MODEL_DEVICE = 'auto'       # 'auto', 'cpu', 'cuda'
TRAINED_MODEL_CLIP_LEN_OVERRIDE = None
TRAINED_MODEL_SMOOTH_WINDOW = 5
TRAINED_MODEL_MIN_UP_LEN = 3
TRAINED_MODEL_INFERENCE_INTERVAL = 1
MODEL_ACTION_CONFIDENCE_MIN = 0.40
MODEL_ACTION_SMOOTH_WINDOW = 7

print('모델/실행 옵션 설정 완료')
print(f'EXERCISE                 = {EXERCISE}')
print(f'VIDEO_VIEW_MODE          = {VIDEO_VIEW_MODE}')
print(f'MODEL_DRIVEN_RECOGNITION = {MODEL_DRIVEN_RECOGNITION}')
print(f'MODEL_DRIVEN_COUNT       = {MODEL_DRIVEN_COUNT}')
print(f'TRAINED_MODEL_REQUIRED   = {TRAINED_MODEL_REQUIRED}')


모델/실행 옵션 설정 완료
EXERCISE                 = 모델 작동안하는중
VIDEO_VIEW_MODE          = side
MODEL_DRIVEN_RECOGNITION = True
MODEL_DRIVEN_COUNT       = True
TRAINED_MODEL_REQUIRED   = True


In [4]:
# ── Cell 3C: 경로 정규화 + 존재 확인 셀 ────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 이후 셀들이 기존 변수명을 그대로 쓰도록 alias를 맞춥니다.
MODEL_PATH = POSE_MODEL_PATH
TRAINED_MODEL_CHECKPOINT = STGCN_CHECKPOINT_PATH

# 폰트 자동 보정
if not os.path.exists(FONT_PATH):
    found = glob.glob('/usr/share/fonts/**/*Nanum*', recursive=True)
    FONT_PATH = found[0] if found else None

required_paths = [
    (USER_VIDEO_PATH,  '사용자영상'),
    (EXPERT_JSON_PATH, '전문가JSON'),
    (MODEL_PATH,       'PoseLandmarker모델'),
]

missing = []
for path, name in required_paths:
    ok = bool(path and os.path.exists(path))
    print(f'{"OK" if ok else "없음"} | {name}: {path}')
    if not ok:
        missing.append((name, path))

if TRAINED_MODEL_CHECKPOINT is None:
    print('OK | ST-GCN checkpoint: None → realtime_stgcn_infer 기본 checkpoint 자동 탐색')
else:
    ckpt_ok = os.path.exists(TRAINED_MODEL_CHECKPOINT)
    print(f'{"OK" if ckpt_ok else "없음"} | ST-GCN checkpoint: {TRAINED_MODEL_CHECKPOINT}')
    if not ckpt_ok:
        missing.append(('ST-GCN checkpoint', TRAINED_MODEL_CHECKPOINT))

print(f'{"OK" if FONT_PATH and os.path.exists(FONT_PATH) else "없음"} | 폰트: {FONT_PATH}')
print(f'출력 폴더 = {OUTPUT_DIR}')

if missing:
    print('\n[주의] 없는 파일이 있습니다. 위 경로 입력 셀에서 경로를 수정하세요:')
    for name, path in missing:
        print(f'  - {name}: {path}')
else:
    print('\n모든 필수 경로 확인 완료')


OK | 사용자영상: C:\Users\kojy1\PycharmProjects\capstone\data\merged_dead_squat.mp4
OK | 전문가JSON: C:\Users\kojy1\PycharmProjects\capstone\expert_dead2.json
OK | PoseLandmarker모델: C:\Users\kojy1\PycharmProjects\capstone\pose_landmarker_lite.task
OK | ST-GCN checkpoint: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\temporal_head_full_ablation\mlp_h128_c16_pw_2.0_ts2_do03_aug1\best_ep013.pt
OK | 폰트: C:\Windows\Fonts\malgun.ttf
출력 폴더 = C:\Users\kojy1\PycharmProjects\capstone\output

모든 필수 경로 확인 완료


In [5]:
# ── Cell 4A: 로컬 런타임(rt) 코드 — 프로젝트 .py import 없음 ─────────
# realtime_compare_side.py에서 이 노트북 실행에 필요한 부분만 셀 안으로 옮긴 버전입니다.
import os
import sys
import json
import math
import time
import types
import urllib.request
from collections import deque

import cv2
import numpy as np

# mediapipe import 보정
for _n in ["mediapipe.tasks.python.genai", "mediapipe.tasks.python.genai.bundler"]:
    sys.modules.setdefault(_n, types.ModuleType(_n))

from mediapipe.tasks.python.vision import pose_landmarker as _pl
from mediapipe.tasks.python.vision.core import vision_task_running_mode as _vtm
from mediapipe.tasks.python.core import base_options as _bo


def _find_image_classes():
    try:
        from mediapipe.tasks.python.vision.core import image as m
        return m.Image, m.ImageFormat
    except (ImportError, AttributeError):
        pass
    try:
        from mediapipe.python._framework_bindings import image as _img
        from mediapipe.python._framework_bindings import image_frame as _imgf
        return _img.Image, _imgf.ImageFormat
    except (ImportError, AttributeError):
        pass
    raise ImportError('mediapipe Image 클래스를 찾을 수 없습니다.')


_MpImage, _ImageFormat = _find_image_classes()

PoseLandmarker        = _pl.PoseLandmarker
PoseLandmarkerOptions = _pl.PoseLandmarkerOptions
PoseLandmarksConns    = _pl.PoseLandmarksConnections.POSE_LANDMARKS
BaseOptions           = _bo.BaseOptions
MpImage               = _MpImage
ImageFormat           = _ImageFormat
RunningMode           = _vtm.VisionTaskRunningMode

_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/'
    'pose_landmarker/pose_landmarker_lite/float16/latest/'
    'pose_landmarker_lite.task'
)

WINDOW_SIZE = 5
EXPERT_ALPHA = 0.7
VISIBILITY_THRESHOLD = 0.6
MIRROR_CAMERA = False


def ensure_model(model_path: str):
    if os.path.exists(model_path):
        print(f'[모델] {os.path.basename(model_path)} 확인 완료')
        return
    print('[모델] 다운로드 중... (~7MB)')
    def _p(n, bs, total):
        if total > 0:
            print(f'\r  {min(n*bs*100//total, 100)}%', end='', flush=True)
    urllib.request.urlretrieve(_MODEL_URL, model_path, _p)
    print(f'\n[모델] 저장 완료 → {model_path}')


def load_expert(exercise: str, json_path: str) -> dict:
    if not os.path.exists(json_path):
        raise RuntimeError(f'{json_path} 없음. expert_preprocess.py 를 먼저 실행하세요.')
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)
    print(f"[전문가] {os.path.basename(json_path)} 로드 완료  ({data['total_frames']}프레임, {data['fps']}fps)")
    return data


def build_landmarker(model_path: str):
    opts = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.4,
        min_pose_presence_confidence=0.4,
        min_tracking_confidence=0.4,
    )
    return PoseLandmarker.create_from_options(opts)


def extract_landmarks(landmarker, bgr: np.ndarray, ts_ms: int):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mp_img = MpImage(image_format=ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_img, ts_ms)
    return result.pose_landmarks[0] if result.pose_landmarks else None


def get_expert_frame_idx(elapsed_ms: float, expert: dict) -> int:
    fps = expert['fps']
    n = expert['total_frames']
    return int((elapsed_ms / 1000.0) * fps) % n


# 기존 노트북 코드가 rt.xxx 형태를 쓰므로 rt 네임스페이스를 셀 안에서 직접 만든다.
rt = types.SimpleNamespace()
rt.EXERCISE = EXERCISE
rt.EXPERT_JSON_PATH = EXPERT_JSON_PATH
rt.MODEL_PATH = MODEL_PATH
rt.MIRROR_CAMERA = MIRROR_CAMERA
rt.WINDOW_SIZE = WINDOW_SIZE
rt.EXPERT_ALPHA = EXPERT_ALPHA
rt.VISIBILITY_THRESHOLD = VISIBILITY_THRESHOLD
rt.PoseLandmarksConns = PoseLandmarksConns
rt.ensure_model = ensure_model
rt.load_expert = load_expert
rt.build_landmarker = build_landmarker
rt.extract_landmarks = extract_landmarks
rt.get_expert_frame_idx = get_expert_frame_idx
rt.EXERCISE_METRICS = {'squat': [], 'deadlift': [], 'bench': [], 'benchpress': []}
rt.DELTA_THRESHOLDS = {}
rt.DELTA_TO_LM = {}
rt.DISPLAY_LABELS = {}
rt.METRIC_LABELS = rt.DISPLAY_LABELS

print('로컬 rt 런타임 준비 완료: realtime_compare_side.py import 없음')
print(f'  WINDOW_SIZE          = {rt.WINDOW_SIZE}')
print(f'  VISIBILITY_THRESHOLD = {rt.VISIBILITY_THRESHOLD}')
print(f'  EXPERT_ALPHA         = {rt.EXPERT_ALPHA}')


로컬 rt 런타임 준비 완료: realtime_compare_side.py import 없음
  WINDOW_SIZE          = 5
  VISIBILITY_THRESHOLD = 0.6
  EXPERT_ALPHA         = 0.7


In [6]:
# ── Cell 4B: feedback_overlay 코드 내장 — 프로젝트 .py import 없음 ─────────
"""
feedback_overlay.py
===================
운동 보조 시스템 - 시각 피드백 레이어 모듈

설계 원칙:
  1. 떨림 방지: 1euro filter (단순 이동평균보다 지연 적음)
  2. 비교 명확성: 각도/비율 기반, 절대좌표 최소화
  3. 인지 부하 최소화: 레이어 독립적, 최대 2~3개 권장
  4. 타이밍: rep 진행 중 시각만, rep 완료 후 텍스트

사용법:
    renderer = FeedbackRenderer(fps=30)
    renderer.enable('rom', 'heatmap')

    # 메인 루프:
    renderer.render(frame, raw_metrics, expert_metrics,
                    deltas, lms, connections, metric_to_lm)

권장 조합:
    초보자 → rom + heatmap
    중급자 → heatmap + depth
    숙련자 → heatmap 단독
"""

import cv2
import numpy as np
import math
import os
from collections import deque
from typing import Dict, List, Optional, Tuple

# ── PIL 한글 지원 ─────────────────────────────────────────────
try:
    from PIL import Image as _PilImg, ImageDraw as _ImageDraw, ImageFont as _PilFont
    _FONTS = [
        "C:/Windows/Fonts/malgun.ttf",
        "C:/Windows/Fonts/gulim.ttc",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    ]
    _fp = next((p for p in _FONTS if os.path.exists(p)), None)
    def _pf(sz): return _PilFont.truetype(_fp, sz) if _fp else _PilFont.load_default()
    _FSS = _pf(13)   # small
    _PIL_OK = True
except Exception:
    _PIL_OK = False

def _put_kr(frame, text, xy, size=13, rgb=(200,200,200)):
    """PIL로 한글 텍스트 렌더링. PIL 없으면 cv2 fallback."""
    if not _PIL_OK:
        cv2.putText(frame, text, xy, cv2.FONT_HERSHEY_SIMPLEX,
                    0.38, (int(rgb[2]),int(rgb[1]),int(rgb[0])), 1, cv2.LINE_AA)
        return
    pil = _PilImg.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    fnt = _pf(size)
    _ImageDraw.Draw(pil).text(xy, text, font=fnt, fill=tuple(rgb))
    cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR, frame)


# ================================================================
# [1] 1euro Filter — 떨림 방지 핵심
# ================================================================
# 논문: Casiez et al. (2012) CHI
#
# 핵심: 속도에 따라 스무딩 강도를 동적으로 조절
#   - 정지 시 (준비 자세): 강하게 스무딩 → 깜빡임 제거
#   - 동작 시 (하강/상승): 약하게 스무딩 → 지연 없음
#
# 비교:
#   이동평균 (window=7):  빠른 동작 시 7프레임 지연 발생
#   1euro filter:         정지=안정, 동작=반응 빠름
#
# 실제 효과 (MediaPipe 기준):
#   raw: ±5~8px 진동
#   1euro (min_cutoff=1.0, beta=0.007): ±1~2px

class OneEuroFilter:
    def __init__(self, freq=30.0, min_cutoff=1.0,
                 beta=0.007, d_cutoff=1.0):
        self.freq = freq
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self._x = None
        self._dx = 0.0

    def _alpha(self, cutoff):
        tau = 1.0 / (2 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / (1.0 / self.freq))

    def __call__(self, x):
        if self._x is None:
            self._x = x
            return x
        dx = (x - self._x) * self.freq
        a_d = self._alpha(self.d_cutoff)
        self._dx = a_d * dx + (1 - a_d) * self._dx
        cutoff = self.min_cutoff + self.beta * abs(self._dx)
        a = self._alpha(cutoff)
        self._x = a * x + (1 - a) * self._x
        return self._x

    def reset(self):
        self._x = None
        self._dx = 0.0


class PoseFilter:
    """관절 좌표/각도 전체에 1euro filter 적용"""
    def __init__(self, freq=30.0, min_cutoff=1.0, beta=0.007, fps=None):
        self._f: Dict[str, OneEuroFilter] = {}
        self._freq = fps if fps is not None else freq
        self._mc = min_cutoff
        self._b = beta

    def _get(self, key):
        if key not in self._f:
            self._f[key] = OneEuroFilter(self._freq, self._mc, self._b)
        return self._f[key]

    def fv(self, key, val):
        """단일 값 필터링"""
        return self._get(key)(val)

    def fp(self, key, x, y):
        """2D 포인트 필터링"""
        return (self._get(key+'_x')(x), self._get(key+'_y')(y))

    def fm(self, metrics: dict) -> dict:
        """지표 딕셔너리 전체 필터링"""
        return {k: self.fv(k, v) for k, v in metrics.items()}

    def reset(self):
        for f in self._f.values():
            f.reset()


# ================================================================
# [2] ROM 게이지 — 가동범위 % 시각화
# ================================================================
# 위치: 영상 왼쪽 세로 바
#
# 표시 내용:
#   [사용자 현재 깊이 %] 채움 바
#   [전문가 현재 위치]   하늘색 가로선 (E 표시)
#   [목표 깊이]          초록 점선 (goal 표시)
#
# 색상:
#   목표 미달: 파란 계열 (차가움 → 더 내려가라)
#   목표 달성: 초록 (OK)
#
# 떨림 대응:
#   - 무릎 각도 → 1euro 스무딩 후 % 변환
#   - % 자체도 3프레임 이동평균 (미세 진동 추가 제거)
#   → 효과: 바가 부드럽게 움직임, 경계에서 깜빡임 없음

class ROMGauge:
    def __init__(self, pf: PoseFilter,
                 angle_min=90.0, angle_max=170.0, target_pct=80.0):
        self._pf = pf
        self._amin = angle_min
        self._amax = angle_max
        self._tgt = target_pct
        self._buf = deque(maxlen=3)

    def _to_pct(self, angle):
        return max(0.0, min(100.0,
            (self._amax - angle) / (self._amax - self._amin) * 100.0))

    def draw(self, frame, user_angle_raw,
             expert_angle=None, expert_min_angle=None,
             x=14, y=70, bar_h=200, bar_w=14):
        H, W = frame.shape[:2]
        s = self._pf.fv('rom', user_angle_raw)
        self._buf.append(self._to_pct(s))
        pct = float(np.mean(self._buf))

        # 배경
        ov = frame.copy()
        cv2.rectangle(ov, (x, y), (x+bar_w, y+bar_h), (25,25,25), -1)

        # 채움
        fill_h = int(bar_h * pct / 100.0)
        for i in range(fill_h):
            r = i / bar_h
            if r < 0.4:   col = (80,200,80)
            elif r < 0.7: col = (0,180,230)
            else:         col = (0,120,255)
            cv2.line(ov, (x, y+bar_h-i), (x+bar_w, y+bar_h-i), col, 1)
        cv2.addWeighted(ov, 0.8, frame, 0.2, 0, frame)
        cv2.rectangle(frame, (x,y), (x+bar_w,y+bar_h), (70,70,70), 1)

        # 목표선
        t_ang = expert_min_angle or (
            self._amax - (self._amax-self._amin)*self._tgt/100.0)
        t_pct = self._to_pct(t_ang)
        t_y = y + int(bar_h*(1-t_pct/100.0))
        for xi in range(x-2, x+bar_w+3, 4):
            cv2.line(frame, (xi,t_y),(min(xi+2,x+bar_w+3),t_y),
                     (80,220,80), 1)
        cv2.putText(frame, 'goal', (x+bar_w+3, t_y+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, (80,220,80),
                    1, cv2.LINE_AA)

        # 전문가 현재 마커
        if expert_angle is not None:
            ey = y + int(bar_h*(1-self._to_pct(expert_angle)/100.0))
            cv2.line(frame,(x-3,ey),(x+bar_w+3,ey),(0,200,255),1)
            cv2.putText(frame,'E',(x+bar_w+3,ey+4),
                        cv2.FONT_HERSHEY_SIMPLEX,0.3,(0,200,255),
                        1,cv2.LINE_AA)

        # % 텍스트 (배경 박스 추가)
        color = (80,220,80) if pct>=t_pct else (120,140,255)
        txt = f'{int(pct)}%'
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 1)
        cv2.rectangle(frame, (x-2, y-18), (x+tw+2, y-2), (20,20,20), -1)
        cv2.putText(frame, txt, (x, y-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, color,
                    1, cv2.LINE_AA)
        if pct >= t_pct:
            cv2.rectangle(frame, (x-2, y+bar_h+2), (x+22, y+bar_h+18),
                          (20,20,20), -1)
            cv2.putText(frame, 'OK', (x, y+bar_h+15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                        (80,220,80), 1, cv2.LINE_AA)


# ================================================================
# [3] Delta 히트맵 — 관절 색상 인코딩
# ================================================================
# 기존 이진(빨강/흰색) 방식 문제:
#   - 임계값 경계에서 깜빡임 (jitter + threshold 결합)
#   - "얼마나 틀렸는지" 정보 없음
#
# 연속 색상:
#   초록(delta=0) → 노랑(delta=임계/2) → 빨강(delta>=임계)
#
# 떨림 대응:
#   - delta를 1euro로 스무딩 후 색상 계산
#   - 색상이 연속적이라 미세 진동이 있어도 눈에 안 띔
#   - (이진 방식은 진동이 색상 전환으로 바로 보임)

class DeltaHeatmap:
    def __init__(self, pf: PoseFilter):
        self._pf = pf

    @staticmethod
    def _to_color(norm):
        if norm < 0.5:
            t = norm * 2.0
            return (0, 200, int(t*255))   # 초록→노랑
        else:
            t = (norm-0.5)*2.0
            return (0, int((1-t)*200), 255)  # 노랑→빨강

    def draw_skeleton(self, frame, lms, deltas,
                      metric_to_lm, connections,
                      threshold=15.0):
        H, W = frame.shape[:2]

        # 관절번호 → 스무딩 delta 매핑
        jd: Dict[int, float] = {}
        for metric, raw_d in deltas.items():
            s = self._pf.fv(f'hm_{metric}', abs(raw_d))
            norm = min(1.0, s / threshold)
            for idx in metric_to_lm.get(metric, []):
                jd[idx] = max(jd.get(idx, 0.0), norm)

        # 뼈대
        for conn in connections:
            a, b = lms[conn.start], lms[conn.end]
            if a.visibility < 0.3 or b.visibility < 0.3:
                continue
            avg = (jd.get(conn.start,0)+jd.get(conn.end,0))/2
            col = self._to_color(avg)
            cv2.line(frame,
                     (int(a.x*W),int(a.y*H)),
                     (int(b.x*W),int(b.y*H)),
                     col, 2, cv2.LINE_AA)

        # 관절
        for i, lm in enumerate(lms):
            if lm.visibility < 0.3:
                continue
            col = self._to_color(jd.get(i,0.0))
            px, py = int(lm.x*W), int(lm.y*H)
            cv2.circle(frame,(px,py),5,col,-1,cv2.LINE_AA)
            cv2.circle(frame,(px,py),5,(0,0,0),1,cv2.LINE_AA)


# ================================================================
# [4] 깊이선 — 수평 기준선 비교
# ================================================================
# 전문가 최저점 힙 y좌표를 점선으로 표시
# 사용자 현재 힙을 실선으로 표시
#
# 왜 힙 y좌표?:
#   무릎각도는 "숫자", 힙 y는 "선으로 위치 표현" → 더 직관적
#   PT 현장에서 "저 테이프까지 내려와" 방식과 동일한 인지
#
# 떨림 대응:
#   힙 y를 1euro로 스무딩. min_cutoff=0.5로 낮춰서
#   정지 시 더 강하게 스무딩 → 선이 안 흔들림

class DepthLine:
    def __init__(self, pf: PoseFilter):
        self._pf = pf
        self._target_y = None

    def set_expert_target(self, y_norm, frame_h):
        self._target_y = int(y_norm * frame_h)

    def draw(self, frame, user_hip_y_raw):
        H, W = frame.shape[:2]
        s_y = self._pf.fv('dl_hip', user_hip_y_raw)
        uy = int(s_y * H)

        # 사용자 현재 힙 (실선, 연한 회색)
        cv2.line(frame, (int(W*0.08),uy), (int(W*0.42),uy),
                 (150,150,150), 1, cv2.LINE_AA)

        if self._target_y is None:
            return

        ty = self._target_y
        # 전문가 목표 (점선, 하늘색)
        dash = 8
        for xi in range(int(W*0.08), int(W*0.42), dash*2):
            cv2.line(frame,(xi,ty),(min(xi+dash,int(W*0.42)),ty),
                     (0,200,255),1,cv2.LINE_AA)

        # 남은 거리 화살표 (아직 못 내려온 경우)
        if uy < ty:
            mx = int(W*0.06)
            cv2.arrowedLine(frame,(mx,uy+4),(mx,ty-4),
                            (0,180,255),1,cv2.LINE_AA,tipLength=0.25)
        else:
            cv2.putText(frame,'depth OK',
                        (int(W*0.08),ty-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.33,
                        (80,220,80),1,cv2.LINE_AA)

    def reset(self):
        pass


# ================================================================
# [5] 지표 비교 바 — HUD 숫자를 시각화로 보완
# ================================================================
# 기존: "무릎 각도 109.3d (+32.1)"
# 개선: 전문가(파란 바) / 사용자(흰→빨간 바) 나란히
#
# 위치: 기존 HUD 숫자 옆에 보조로 배치
# 역할: HUD 대체가 아닌 보완 — 운동 중 빠른 확인용
#
# 떨림 대응:
#   - 1euro로 스무딩된 값으로 바 길이 계산
#   - 바 길이 자체도 deque(4) 평균 → 미세 진동 제거

class MetricBar:
    def __init__(self, pf: PoseFilter):
        self._pf = pf
        self._bufs: Dict[str, deque] = {}

    def _sb(self, key, val):
        if key not in self._bufs:
            self._bufs[key] = deque(maxlen=4)
        self._bufs[key].append(val)
        return float(np.mean(self._bufs[key]))

    def draw_one(self, frame, x, y, bar_w,
                 u_val, e_val, val_max, label):
        u_r = min(1.0, u_val/val_max)
        e_r = min(1.0, e_val/val_max)
        ub  = int(self._sb(label+'u', bar_w*u_r))
        eb  = int(self._sb(label+'e', bar_w*e_r))
        diff = abs(u_val - e_val)
        dr   = min(1.0, diff/20.0)

        _put_kr(frame, label, (x, y+1), size=13, rgb=(150,150,150))
        bx = x+36
        # 전문가 (파란 계열)
        if eb: cv2.rectangle(frame,(bx,y+2),(bx+eb,y+7),(160,90,0),-1)
        # 사용자 (흰→빨강)
        uc = (int(180*(1-dr)), int(180*(1-dr)), 180)
        if ub: cv2.rectangle(frame,(bx,y+9),(bx+ub,y+14),uc,-1)
        # 차이 수치 (클 때만)
        if diff > 12:
            s = '+' if u_val>e_val else '-'
            cv2.putText(frame, f'{s}{diff:.0f}',
                        (bx+bar_w+3,y+10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.29,
                        (80,110,255),1,cv2.LINE_AA)

    def draw_all(self, frame, u_metrics, e_metrics,
                 keys, labels, x, y, bar_w=100,
                 val_max=180.0, row_h=18):
        H = frame.shape[0]
        for i, k in enumerate(keys):
            ry = y + i*row_h
            if ry + row_h > H-10:
                break
            s_u = self._pf.fv(f'mb_{k}', u_metrics.get(k,0))
            self.draw_one(frame, x, ry, bar_w,
                          s_u, e_metrics.get(k,0),
                          val_max, labels.get(k,k[:4]))


# ================================================================
# [6] FeedbackRenderer — 통합 렌더러
# ================================================================

class FeedbackRenderer:
    def __init__(self, fps=30.0):
        self.pf        = PoseFilter(fps=fps, min_cutoff=1.0, beta=0.007)
        self.rom       = ROMGauge(self.pf)
        self.heatmap   = DeltaHeatmap(self.pf)
        self.depth_line = DepthLine(self.pf)
        self.metricbar = MetricBar(self.pf)
        self._on       = {'heatmap', 'rom'}   # 기본 활성화

    def enable(self, *layers):
        for l in layers: self._on.add(l)

    def disable(self, *layers):
        for l in layers: self._on.discard(l)

    def is_on(self, layer):
        return layer in self._on

    def reset(self):
        self.pf.reset()

    def render(self, frame, user_metrics_raw, expert_metrics,
               deltas, lms, connections, metric_to_lm,
               exercise='squat'):
        if lms is None:
            return frame

        # 렌더링 순서: 배경 → 스켈레톤 → 사이드 UI
        if 'depth' in self._on:
            hy = (lms[23].y + lms[24].y) / 2
            self.depth_line.draw(frame, hy)

        if 'heatmap' in self._on:
            self.heatmap.draw_skeleton(
                frame, lms, deltas, metric_to_lm,
                connections, threshold=15.0)

        if 'rom' in self._on:
            self.rom.draw(
                frame,
                user_metrics_raw.get('knee_angle', 170.0),
                expert_metrics.get('knee_angle', None))

        if 'metricbar' in self._on:
            H, W = frame.shape[:2]
            km = {'squat':   ['knee_angle','hip_angle','spine_lean','foot_width'],
                  'pushup':  ['elbow_angle','body_angle','spine_lean'],
                  'deadlift':['hip_angle','knee_angle','spine_lean']}
            lb = {'knee_angle':'무릎','hip_angle':'힙  ',
                  'spine_lean':'척추','foot_width':'발폭',
                  'elbow_angle':'팔꿈','body_angle':'몸통'}
            self.metricbar.draw_all(
                frame, user_metrics_raw, expert_metrics,
                km.get(exercise, ['knee_angle','hip_angle']),
                lb, x=W-185, y=215, bar_w=105)

        return frame


# ================================================================
# [7] KneeToeGuard — 무릎-발끝 기준선
# ================================================================
# 기획 이유:
#   스쿼트에서 "무릎이 발끝을 넘어가는 것"이 가장 흔한 오류.
#   논문(ScienceDirect 2025)에서 knee-over-toe가 핵심 감지 대상.
#   수직선 하나라 구현이 단순하고 떨림에 강함.
#   사용자는 "저 선을 넘으면 안 된다"는 규칙을 즉각 이해.
#
# 구현 결정:
#   - 발목 x좌표 → 수직 점선 (전신에 걸쳐)
#   - 무릎 x가 선을 넘으면 선 빨간색 + 무릎 위에 경고 원
#   - 1euro로 발목/무릎 x 스무딩 → 선이 흔들리지 않음
#   - 전문가는 다른 색(파란 점선)으로 비교 기준 제공
#
# 평가:
#   ✅ 측면 촬영에서 매우 직관적
#   ✅ 숫자/텍스트 없이 즉각 인지
#   ⚠ 정면 촬영에서는 x좌표 의미 없음 (측면 전용)
#   ⚠ 좌우 다리 평균 쓰면 비대칭 오류 놓칠 수 있음

class KneeToeGuard:
    def __init__(self, pf: PoseFilter):
        self._pf = pf

    def draw(self, frame, lms,
             expert_ankle_x_norm: Optional[float] = None) -> None:
        """
        lms: 사용자 MediaPipe 랜드마크
        expert_ankle_x_norm: 전문가 발목 x (0~1), 없으면 비교선 안 그림
        """
        H, W = frame.shape[:2]

        # 1euro 스무딩 (좌우 각각)
        l_ankle_x = self._pf.fv('ktg_l_ankle_x', lms[27].x)
        r_ankle_x = self._pf.fv('ktg_r_ankle_x', lms[28].x)
        l_knee_x  = self._pf.fv('ktg_l_knee_x',  lms[25].x)
        r_knee_x  = self._pf.fv('ktg_r_knee_x',  lms[26].x)
        l_knee_y  = self._pf.fv('ktg_l_knee_y',  lms[25].y)

        # 더 심한 쪽 선택
        l_diff = (l_knee_x - l_ankle_x) * W
        r_diff = (r_knee_x - r_ankle_x) * W
        if abs(l_diff) >= abs(r_diff):
            ankle_x_px = int(l_ankle_x * W)
            knee_x_px  = int(l_knee_x  * W)
            knee_y_px  = int(l_knee_y  * H)
            diff_px    = l_diff
        else:
            ankle_x_px = int(r_ankle_x * W)
            knee_x_px  = int(r_knee_x  * W)
            knee_y_px  = int(l_knee_y  * H)
            diff_px    = r_diff

        shoulder_w_px = abs(lms[11].x - lms[12].x) * W
        threshold_px  = max(shoulder_w_px * 0.05, 8.0)  # 최소 8px
        is_over = diff_px > threshold_px

        # ── 수직선 (두께 2) ──
        line_color = (0, 60, 200) if is_over else (80, 200, 80)
        dash = 8
        for yi in range(int(H * 0.05), int(H * 0.95), dash * 2):
            cv2.line(frame,
                     (ankle_x_px, yi),
                     (ankle_x_px, min(yi+dash, H)),
                     line_color, 2, cv2.LINE_AA)

        # ── 전문가 기준선 (하늘색, 있을 때만) ──
        if expert_ankle_x_norm is not None:
            ex_ax = int(expert_ankle_x_norm * W)
            for yi in range(int(H*0.2), int(H*0.8), dash*2):
                cv2.line(frame,(ex_ax,yi),
                         (ex_ax, min(yi+dash,H)),
                         (200,180,0),1,cv2.LINE_AA)

        # ── 무릎 경고 (넘었을 때만) ──
        if is_over:
            severity = min(1.0, diff_px / (shoulder_w_px * 0.2 + 1e-6))
            radius = int(10 + severity * 8)
            cv2.circle(frame,(knee_x_px,knee_y_px),
                       radius,(0,0,220),2,cv2.LINE_AA)
            d = radius // 2
            cv2.line(frame,(knee_x_px-d,knee_y_px-d),
                     (knee_x_px+d,knee_y_px+d),(0,0,220),2,cv2.LINE_AA)
            cv2.line(frame,(knee_x_px+d,knee_y_px-d),
                     (knee_x_px-d,knee_y_px+d),(0,0,220),2,cv2.LINE_AA)
            # 텍스트 배경 박스
            txt = f'+{int(diff_px)}px'
            (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.36, 1)
            tx, ty = knee_x_px+8, knee_y_px-8
            cv2.rectangle(frame,(tx-2,ty-th-2),(tx+tw+2,ty+2),(20,20,20),-1)
            cv2.putText(frame,txt,(tx,ty),
                        cv2.FONT_HERSHEY_SIMPLEX,0.36,
                        (80,100,255),1,cv2.LINE_AA)


# ================================================================
# [8] JointArcGauge — 관절 위 원호 게이지
# ================================================================
# 기획 이유:
#   숫자(109.3°)를 운동 중에 읽기 어려움.
#   부채꼴(arc) 면적으로 각도를 표현하면 즉각 인지 가능.
#   전문가(외부 파란 호)와 사용자(내부 흰 채움)를 겹쳐서
#   면적 차이로 오차를 시각화.
#
# 구현 결정:
#   - 무릎 관절 위에 반원 형태 게이지
#   - 내부 채움(흰): 사용자 각도
#   - 외부 테두리 호(파란): 전문가 각도
#   - 차이가 크면 호가 빨간색으로 변함
#   - 각도를 0~180도 → 호의 0~180도로 매핑
#
# 1euro 적용:
#   각도를 스무딩 후 호 그리기
#   → 부채꼴이 부드럽게 변함, 경계에서 깜빡임 없음
#
# 평가:
#   ✅ 숫자 없이 면적으로 직관 표현
#   ✅ 전문가/사용자 차이를 겹쳐서 한눈에
#   ⚠ 관절이 화면 밖이면 안 보임
#   ⚠ 여러 관절에 동시 그리면 복잡 → 무릎 하나만 권장

class JointArcGauge:
    def __init__(self, pf: PoseFilter):
        self._pf = pf

    def draw(self, frame, joint_px: Tuple[int,int],
             user_angle_raw: float,
             expert_angle: float,
             key: str,
             radius: int = 28) -> None:
        """
        joint_px    : (x, y) 픽셀 좌표
        user_angle  : 사용자 각도 (예: 무릎각도 90~170)
        expert_angle: 전문가 각도
        radius      : 게이지 반지름 (px)
        """
        # 1euro 스무딩
        s_user = self._pf.fv(key + '_arc_u', user_angle_raw)
        # 전문가는 이미 스무딩된 값이므로 그대로 사용

        jx, jy = int(joint_px[0]), int(joint_px[1])

        # 각도 → 호 각도 변환 (OpenCV ellipse: 0=오른쪽, 반시계)
        # 무릎각도 90도 → 게이지 90% 채움 (많이 구부림 = 많이 채움)
        # 무릎각도 170도 → 게이지 10% 채움 (거의 안 구부림)
        angle_min, angle_max = 70.0, 175.0

        def to_sweep(angle):
            ratio = 1.0 - (angle - angle_min) / (angle_max - angle_min)
            return max(0.0, min(1.0, ratio)) * 180.0  # 반원

        user_sweep   = to_sweep(s_user)
        expert_sweep = to_sweep(expert_angle)
        diff = abs(s_user - expert_angle)

        # ── 배경 반원 (어두운 회색) ──
        overlay = frame.copy()
        cv2.ellipse(overlay, (jx, jy),
                    (radius, radius), 180, 0, 180,
                    (40, 40, 40), -1)
        cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

        # ── 사용자 채움 (흰→빨강) ──
        diff_ratio = min(1.0, diff / 30.0)
        u_color = (
            int(200 * (1 - diff_ratio)),
            int(200 * (1 - diff_ratio)),
            200
        )
        if user_sweep > 1:
            cv2.ellipse(frame, (jx, jy),
                        (radius, radius), 180, 0, int(user_sweep),
                        u_color, -1, cv2.LINE_AA)

        # ── 전문가 외부 호 (파란색) ──
        exp_color = (200, 120, 0) if diff < 15 else (80, 80, 255)
        if expert_sweep > 1:
            cv2.ellipse(frame, (jx, jy),
                        (radius+4, radius+4), 180, 0, int(expert_sweep),
                        exp_color, 2, cv2.LINE_AA)

        # ── 테두리 ──
        cv2.ellipse(frame, (jx, jy),
                    (radius, radius), 180, 0, 180,
                    (80, 80, 80), 1, cv2.LINE_AA)

        # ── 차이 텍스트 (배경 박스 포함) ──
        if diff > 12:
            txt = f'{diff:.0f}deg'
            (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.36, 1)
            tx = jx + radius + 6
            ty = jy + 4
            cv2.rectangle(frame,(tx-2,ty-th-2),(tx+tw+2,ty+2),(20,20,20),-1)
            cv2.putText(frame, txt, (tx, ty),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.36,
                        (80, 120, 255) if diff > 20 else (160, 160, 160),
                        1, cv2.LINE_AA)


# ================================================================
# [9] FormScoreBar — 실시간 자세 점수 바
# ================================================================
# 기획 이유:
#   논문(MDPI 2026)에서 "form accuracy percentage" 단일 지표가
#   216명 실험에서 가장 효과적인 피드백으로 검증됨.
#   여러 지표를 하나의 점수로 압축 → 인지 부하 최소화.
#   사용자는 "지금 몇 점인지" 하나만 보면 됨.
#
# 구현 결정:
#   - 각 지표별 delta/threshold를 정규화
#   - 가중치 합산 → 0~100% 점수
#   - 화면 우상단에 가로 바 + 숫자
#   - 색상: 80% 이상=초록, 60~80%=노랑, 60% 미만=빨강
#   - 1euro로 점수 스무딩 → 바가 안 떨림
#
# 가중치 설계 (스쿼트 기준):
#   무릎각도: 30% (핵심)
#   힙각도:   25% (핵심)
#   척추기울기: 20%
#   발폭:     15%
#   무릎발끝:  10%
#
# 평가:
#   ✅ 단순, 직관적 (게임 점수처럼)
#   ✅ 1euro로 안정적
#   ✅ 논문에서 효과 검증됨
#   ⚠ "왜 점수가 낮은지" 모름 → 다른 레이어와 병행 필요
#   ⚠ 가중치가 종목마다 달라야 함

class FormScoreBar:
    WEIGHTS = {
        'squat':    {'knee_angle':0.30,'hip_angle':0.25,
                     'spine_lean':0.20,'foot_width':0.15,'knee_align':0.10},
        'deadlift': {'hip_angle':0.35,'knee_angle':0.20,
                     'spine_lean':0.30,'foot_width':0.15},
        'pushup':   {'elbow_angle':0.35,'body_angle':0.30,
                     'spine_lean':0.25,'grip_width':0.10},
    }
    THRESHOLDS = {
        'knee_angle':12.0,'hip_angle':12.0,'spine_lean':8.0,
        'foot_width':0.15,'knee_align':0.1,'elbow_angle':12.0,
        'body_angle':10.0,'grip_width':0.15,
    }

    def __init__(self, pf: PoseFilter):
        self._pf  = pf
        self._buf = deque(maxlen=5)  # 추가 평탄화

    def _calc_score(self, deltas: dict, exercise: str) -> float:
        weights = self.WEIGHTS.get(exercise, self.WEIGHTS['squat'])
        total_w, weighted_ok = 0.0, 0.0
        for key, w in weights.items():
            delta = abs(deltas.get(key, 0.0))
            thr   = self.THRESHOLDS.get(key, 15.0)
            # dead zone: 임계값의 20% 이내는 완전 OK로 처리
            dead  = thr * 0.2
            if delta <= dead:
                ok_ratio = 1.0
            else:
                ok_ratio = max(0.0, 1.0 - (delta - dead) / (thr - dead))
            weighted_ok += ok_ratio * w
            total_w += w
        return (weighted_ok / total_w * 100.0) if total_w > 0 else 100.0

    def draw(self, frame, deltas: dict, exercise: str = 'squat',
             x: int = None, y: int = 10, bar_w: int = 120) -> None:
        H, W = frame.shape[:2]
        if x is None:
            x = W - bar_w - 10

        # 점수 계산 + 스무딩
        raw_score = self._calc_score(deltas, exercise)
        s_score   = self._pf.fv('form_score', raw_score)
        self._buf.append(s_score)
        score = float(np.mean(self._buf))

        # 색상
        if score >= 80:
            bar_color = (80, 200, 80)
            txt_color = (80, 220, 80)
        elif score >= 60:
            bar_color = (0, 180, 230)
            txt_color = (0, 200, 255)
        else:
            bar_color = (0, 80, 220)
            txt_color = (0, 100, 255)

        # ── 배경 박스 (텍스트 포함 전체) ──
        cv2.rectangle(frame, (x-42, y-2), (x+bar_w+58, y+16),
                      (20,20,20), -1)
        cv2.rectangle(frame, (x-42, y-2), (x+bar_w+58, y+16),
                      (55,55,55), 1)

        # ── 배경 바 ──
        cv2.rectangle(frame, (x, y+1), (x+bar_w, y+13),
                      (45, 45, 45), -1)

        # ── 채움 바 ──
        fill_w = int(bar_w * score / 100.0)
        if fill_w > 0:
            cv2.rectangle(frame, (x, y+1), (x+fill_w, y+13),
                          bar_color, -1)

        # ── 점수 텍스트 ──
        cv2.putText(frame, f'{int(score)}%',
                    (x + bar_w + 5, y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48,
                    txt_color, 1, cv2.LINE_AA)

        # ── 레이블 ──
        cv2.putText(frame, 'FORM',
                    (x - 38, y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.36,
                    (160, 160, 160), 1, cv2.LINE_AA)


# ================================================================
# [10] FeedbackRenderer 업데이트 (3개 레이어 추가)
# ================================================================

class FeedbackRenderer:
    def __init__(self, fps=30.0):
        self.pf         = PoseFilter(fps=fps, min_cutoff=1.0, beta=0.007)
        self.rom        = ROMGauge(self.pf)
        self.heatmap    = DeltaHeatmap(self.pf)
        self.depth_line = DepthLine(self.pf)
        self.metricbar  = MetricBar(self.pf)
        self.ktg        = KneeToeGuard(self.pf)      # 신규
        self.arc        = JointArcGauge(self.pf)     # 신규
        self.score_bar  = FormScoreBar(self.pf)      # 신규
        self._on        = {'heatmap', 'rom', 'score_bar'}  # 기본 활성화

    def enable(self, *layers):
        for l in layers: self._on.add(l)

    def disable(self, *layers):
        for l in layers: self._on.discard(l)

    def is_on(self, layer):
        return layer in self._on

    def reset(self):
        self.pf.reset()

    def render(self, frame, user_metrics_raw, expert_metrics,
               deltas, lms, connections, metric_to_lm,
               exercise='squat'):
        if lms is None:
            return frame

        H, W = frame.shape[:2]

        # ── 배경 레이어 ──
        if 'depth' in self._on:
            hy = (lms[23].y + lms[24].y) / 2
            self.depth_line.draw(frame, hy)

        if 'ktg' in self._on:
            # 전문가 발목 x (현재 프레임 기준)
            ex_ankle_x = expert_metrics.get('ankle_x_norm', None)
            self.ktg.draw(frame, lms, ex_ankle_x)

        # ── 스켈레톤 레이어 ──
        if 'heatmap' in self._on:
            self.heatmap.draw_skeleton(
                frame, lms, deltas, metric_to_lm,
                connections, threshold=15.0)

        # ── 관절 게이지 ──
        if 'arc' in self._on:
            # 무릎 관절 위치
            lknee_x = self.pf.fv('arc_lkx', lms[25].x) * W
            lknee_y = self.pf.fv('arc_lky', lms[25].y) * H
            self.arc.draw(
                frame,
                joint_px=(lknee_x, lknee_y),
                user_angle_raw=user_metrics_raw.get('knee_angle', 170.0),
                expert_angle=expert_metrics.get('knee_angle', 170.0),
                key='lknee',
                radius=26
            )

        # ── 사이드 UI ──
        if 'rom' in self._on:
            self.rom.draw(
                frame,
                user_metrics_raw.get('knee_angle', 170.0),
                expert_metrics.get('knee_angle', None))

        if 'score_bar' in self._on:
            self.score_bar.draw(frame, deltas, exercise,
                                x=W-130, y=8, bar_w=110)

        if 'metricbar' in self._on:
            km = {'squat':   ['knee_angle','hip_angle','spine_lean','foot_width'],
                  'pushup':  ['elbow_angle','body_angle','spine_lean'],
                  'deadlift':['hip_angle','knee_angle','spine_lean']}
            lb = {'knee_angle':'무릎','hip_angle':'힙  ',
                  'spine_lean':'척추','foot_width':'발폭',
                  'elbow_angle':'팔꿈','body_angle':'몸통'}
            self.metricbar.draw_all(
                frame, user_metrics_raw, expert_metrics,
                km.get(exercise, ['knee_angle','hip_angle']),
                lb, x=W-185, y=215, bar_w=105)

        return frame


# ================================================================
# [12] PhaseIndicator — 동작 단계 표시
# ================================================================
# 기획:
#   스쿼트는 준비→하강→최저점→상승 4단계로 구성됨.
#   현재 단계를 알면 "지금 뭘 해야 하는지"가 명확해지고
#   단계별로 다른 피드백을 줄 수도 있음.
#   무릎각도 + 변화율로 판단: 각도가 작아지면 하강, 커지면 상승.
#
# 구현:
#   1euro 스무딩된 무릎각도의 프레임 간 변화율로 phase 결정.
#   화면 우측에 4개 블록으로 표시. 현재 단계 강조.
#
# 평가:
#   ✅ 기존 knee_angle만으로 판단 가능, 추가 데이터 없음
#   ✅ 단계별 색상으로 직관적
#   ⚠ 각도 변화율이 작을 때 phase 경계에서 깜빡임 가능
#      → dead zone 적용으로 해결

class PhaseIndicator:
    PHASES = ['준비', '하강', '최저점', '상승']
    COLORS = [
        (120, 120, 120),   # 준비: 회색
        (0,   150, 255),   # 하강: 파란색
        (0,   80,  220),   # 최저점: 진한 파란색
        (80,  220, 80),    # 상승: 초록색
    ]
    DOWN_THR = 115.0   # 최저점 판단 무릎각도 임계값
    DEAD_ZONE = 0.003  # 변화율 dead zone (이하면 정지로 판단)

    def __init__(self, pf: PoseFilter):
        self._pf     = pf
        self._prev   = None
        self._phase  = 0   # 0=준비, 1=하강, 2=최저점, 3=상승
        self._stable = 0   # 연속 같은 phase 프레임 수

    def update(self, knee_angle_raw: float) -> int:
        s = self._pf.fv('phase_knee', knee_angle_raw)
        if self._prev is None:
            self._prev = s
            return self._phase

        d = s - self._prev   # 양수=각도 증가=상승, 음수=각도 감소=하강
        self._prev = s

        if s < self.DOWN_THR:
            new_phase = 2  # 최저점
        elif d < -self.DEAD_ZONE:
            new_phase = 1  # 하강 중
        elif d > self.DEAD_ZONE:
            new_phase = 3  # 상승 중
        else:
            new_phase = 0  # 준비/정지

        if new_phase == self._phase:
            self._stable = min(self._stable + 1, 10)
        else:
            self._stable -= 2
            if self._stable <= 0:
                self._stable = 0
                self._phase  = new_phase

        return self._phase

    def draw(self, frame, phase: int,
             x: int = None, y: int = 30,
             block_h: int = 18, block_w: int = 52) -> None:
        H, W = frame.shape[:2]
        if x is None:
            x = W - block_w - 8

        for i, (name, col) in enumerate(zip(self.PHASES, self.COLORS)):
            ry = y + i * (block_h + 3)
            active = (i == phase)

            # 배경
            bg = col if active else (25, 25, 25)
            cv2.rectangle(frame, (x, ry), (x+block_w, ry+block_h), bg, -1)
            cv2.rectangle(frame, (x, ry), (x+block_w, ry+block_h),
                          col if active else (60,60,60), 1)

            # 텍스트
            txt_col = (255,255,255) if active else (100,100,100)
            _put_kr(frame, name, (x+4, ry+2), size=12, rgb=txt_col)

            # 활성 인디케이터 (왼쪽 바)
            if active:
                cv2.rectangle(frame, (x-4, ry), (x-2, ry+block_h), col, -1)


# ================================================================
# [13] SymmetryBar — 좌우 대칭 바
# ================================================================
# 기획:
#   좌/우 무릎 y좌표 차이로 골반 수평 여부 판단.
#   한쪽으로 치우쳐 앉는 오류를 즉각 감지.
#   중앙 정렬 바로 직관적 표시.
#
# 구현:
#   lms[25].y - lms[26].y (왼무릎 - 오른무릎) 차이를 1euro 스무딩.
#   차이가 클수록 바가 한쪽으로 치우침.
#
# 평가:
#   ✅ 추가 계산 없이 기존 lms만으로 구현
#   ✅ 직관적 (가운데가 좋음을 직관적으로 인지)
#   ⚠ 측면 촬영에서 좌우 무릎 y 차이는 카메라 각도 영향받음
#      → 허용 범위 넓게 설정 (dead zone)

class SymmetryBar:
    def __init__(self, pf: PoseFilter):
        self._pf = pf

    def draw(self, frame, lms,
             x: int = None, y: int = None,
             bar_w: int = 100) -> None:
        H, W = frame.shape[:2]
        if x is None: x = W//2 - bar_w//2
        if y is None: y = H - 30

        l_y = self._pf.fv('sym_ly', lms[25].y)
        r_y = self._pf.fv('sym_ry', lms[26].y)
        diff = l_y - r_y   # 양수 = 왼쪽이 더 낮음

        # 정규화 (-0.05~0.05 → -1~1)
        ratio = max(-1.0, min(1.0, diff / 0.05))
        is_ok = abs(ratio) < 0.3

        # ── 배경 바 ──
        cv2.rectangle(frame, (x, y), (x+bar_w, y+10), (30,30,30), -1)
        cv2.rectangle(frame, (x, y), (x+bar_w, y+10), (60,60,60), 1)

        # ── 채움 (중앙 기준) ──
        cx = x + bar_w//2
        fill = int(abs(ratio) * bar_w//2)
        col = (80,220,80) if is_ok else (0,80,220)
        if ratio > 0:
            cv2.rectangle(frame, (cx, y+1), (cx+fill, y+9), col, -1)
        else:
            cv2.rectangle(frame, (cx-fill, y+1), (cx, y+9), col, -1)

        # 중앙선
        cv2.line(frame, (cx, y-2), (cx, y+12), (180,180,180), 1)

        # 레이블
        _put_kr(frame, 'L', (x-14, y), size=11, rgb=(160,160,160))
        _put_kr(frame, 'R', (x+bar_w+3, y), size=11, rgb=(160,160,160))
        if not is_ok:
            side = 'L' if ratio > 0 else 'R'
            cv2.putText(frame, f'{side} tilt', (x, y-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.32,
                        (0,100,255), 1, cv2.LINE_AA)


# ================================================================
# [14] TempoWave — 각도 변화율 파형
# ================================================================
# 기획:
#   반동(너무 빠른 하강)이나 잠금(너무 느린 상승)을 감지.
#   각도 변화율을 파형으로 표시해서 리듬을 시각화.
#   적정 속도 범위를 점선으로 표시.
#
# 구현:
#   무릎각도 변화율 (d_angle/frame)을 deque(40)에 저장.
#   화면 상단에 작은 파형 그래프로 표시.
#   ±임계값 초과 시 빨간색.
#
# 평가:
#   ✅ 기존 1euro 필터 출력값 사용, 추가 계산 없음
#   ✅ 반동 감지에 효과적
#   ⚠ 적정 속도 임계값이 종목/개인마다 다름 → 상대값 사용

class TempoWave:
    def __init__(self, pf: PoseFilter, buf_size: int = 40):
        self._pf   = pf
        self._prev = None
        self._buf  = deque(maxlen=buf_size)
        self.SPEED_THR = 3.0   # 도/프레임, 이 이상이면 빠름

    def update(self, knee_angle_raw: float) -> float:
        s = self._pf.fv('tempo_knee', knee_angle_raw)
        d = 0.0
        if self._prev is not None:
            d = s - self._prev
        self._prev = s
        self._buf.append(d)
        return d

    def draw(self, frame,
             x: int = None, y: int = 28,
             w: int = 120, h: int = 28) -> None:
        H, W = frame.shape[:2]
        if x is None: x = W//2 - w//2
        n = len(self._buf)
        if n < 2:
            return

        # 배경
        ov = frame.copy()
        cv2.rectangle(ov, (x, y), (x+w, y+h), (20,20,20), -1)
        cv2.addWeighted(ov, 0.7, frame, 0.3, 0, frame)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (50,50,50), 1)

        # 중앙선 (0 속도)
        mid_y = y + h//2
        cv2.line(frame, (x, mid_y), (x+w, mid_y), (70,70,70), 1)

        # 임계값 점선
        thr_px = int(h/2 * self.SPEED_THR / 5.0)
        for xi in range(x, x+w, 6):
            cv2.line(frame, (xi, mid_y-thr_px), (xi+3, mid_y-thr_px),
                     (0,80,150), 1)
            cv2.line(frame, (xi, mid_y+thr_px), (xi+3, mid_y+thr_px),
                     (0,80,150), 1)

        # 파형
        pts = list(self._buf)
        max_val = max(abs(v) for v in pts) if pts else self.SPEED_THR
        max_val = max(max_val, self.SPEED_THR * 0.5)  # 최소 스케일
        scale = (h//2 - 2) / max_val
        for i in range(1, n):
            px0 = x + int((i-1) / n * w)
            px1 = x + int(i     / n * w)
            py0 = mid_y - int(pts[i-1] * scale)
            py1 = mid_y - int(pts[i]   * scale)
            py0 = max(y+1, min(y+h-1, py0))
            py1 = max(y+1, min(y+h-1, py1))
            fast = abs(pts[i]) > self.SPEED_THR
            col  = (0, 60, 220) if fast else (0, 180, 220)
            cv2.line(frame, (px0,py0), (px1,py1), col, 1, cv2.LINE_AA)

        # 레이블
        cv2.putText(frame, 'TEMPO', (x+2, y+h-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.28,
                    (100,100,100), 1, cv2.LINE_AA)


# ================================================================
# [15] ACLRiskScore — ACL 부상 위험도 점수
# ================================================================
# 기획:
#   무릎각도(너무 얕음) + 무릎-발끝 정렬(무릎 전방) +
#   척추 기울기(과도한 앞쏠림)를 결합해서 단일 위험도 점수로 표시.
#   논문(PMC 2020): 이 3가지 조합이 ACL 위험 예측에 유효.
#
# 구현:
#   각 요소를 0~1로 정규화 후 가중 평균.
#   낮을수록 안전 (초록), 높을수록 위험 (빨강).
#
# 평가:
#   ✅ 기존 deltas/metrics만으로 계산 가능
#   ✅ 단일 지표라 인지 부하 없음
#   ⚠ 2D 카메라로는 진정한 valgus 측정 불가,
#     knee_align (x 정렬) 로 근사

class ACLRiskScore:
    def __init__(self, pf: PoseFilter):
        self._pf  = pf
        self._buf = deque(maxlen=5)

    def _calc(self, user_metrics: dict, deltas: dict) -> float:
        # 요소 1: 무릎각도 (너무 얕으면 위험, <120도에서 높아짐)
        knee = user_metrics.get('knee_angle', 170.0)
        r_knee = max(0.0, min(1.0, (130 - knee) / 40.0)) if knee < 130 else 0.0

        # 요소 2: 무릎-발끝 정렬 (knee_align delta)
        ka = abs(deltas.get('knee_align', 0.0))
        r_align = min(1.0, ka / 0.15)

        # 요소 3: 척추 기울기 (spine_lean)
        sl = abs(user_metrics.get('spine_lean', 0.0))
        r_spine = min(1.0, sl / 20.0)

        # 가중 평균 (무릎정렬 50%, 무릎각도 30%, 척추 20%)
        return r_align*0.5 + r_knee*0.3 + r_spine*0.2

    def draw(self, frame, user_metrics: dict, deltas: dict,
             x: int = None, y: int = None) -> None:
        H, W = frame.shape[:2]
        if x is None: x = 14
        if y is None: y = H - 90

        raw = self._calc(user_metrics, deltas)
        s   = self._pf.fv('acl_risk', raw)
        self._buf.append(s)
        risk = float(sum(self._buf)/len(self._buf))

        # 색상: 초록→노랑→빨강
        if risk < 0.4:
            col = (80, 200, 80)
            label = 'SAFE'
        elif risk < 0.7:
            col = (0, 180, 220)
            label = 'CAUTION'
        else:
            col = (0, 60, 220)
            label = 'RISK'

        # 배경
        cv2.rectangle(frame, (x-2, y-2), (x+60, y+28),
                      (20,20,20), -1)
        cv2.rectangle(frame, (x-2, y-2), (x+60, y+28),
                      col, 1)

        # ACL 텍스트
        cv2.putText(frame, 'ACL',
                    (x+2, y+11),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                    (140,140,140), 1, cv2.LINE_AA)
        cv2.putText(frame, label,
                    (x+2, y+24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                    col, 1, cv2.LINE_AA)

        # 위험도 바 (세로)
        bar_h = 24
        fill_h = int(bar_h * risk)
        cv2.rectangle(frame, (x+52, y), (x+58, y+bar_h),
                      (40,40,40), -1)
        if fill_h > 0:
            cv2.rectangle(frame, (x+52, y+bar_h-fill_h),
                          (x+58, y+bar_h), col, -1)


# ================================================================
# FeedbackRenderer 업데이트
# ================================================================

class FeedbackRenderer:
    def __init__(self, fps=30.0):
        self.pf         = PoseFilter(fps=fps, min_cutoff=1.0, beta=0.007)
        self.rom        = ROMGauge(self.pf)
        self.heatmap    = DeltaHeatmap(self.pf)
        self.depth_line = DepthLine(self.pf)
        self.metricbar  = MetricBar(self.pf)
        self.ktg        = KneeToeGuard(self.pf)
        self.arc        = JointArcGauge(self.pf)
        self.score_bar  = FormScoreBar(self.pf)
        self.phase      = PhaseIndicator(self.pf)   # 신규
        self.symmetry   = SymmetryBar(self.pf)      # 신규
        self.tempo      = TempoWave(self.pf)        # 신규
        self.acl        = ACLRiskScore(self.pf)     # 신규
        self._on        = {'heatmap', 'rom', 'score_bar'}
        self._cur_phase = 0

    def enable(self, *layers):
        for l in layers: self._on.add(l)

    def disable(self, *layers):
        for l in layers: self._on.discard(l)

    def is_on(self, layer):
        return layer in self._on

    def get_phase(self) -> int:
        return self._cur_phase

    def reset(self):
        self.pf.reset()
        self.tempo._prev = None
        self.phase._prev = None

    def render(self, frame, user_metrics_raw, expert_metrics,
               deltas, lms, connections, metric_to_lm,
               exercise='squat'):
        if lms is None:
            return frame

        H, W = frame.shape[:2]

        # ── Phase 업데이트 (항상) ──
        knee_raw = user_metrics_raw.get('knee_angle', 170.0)
        self._cur_phase = self.phase.update(knee_raw)
        self.tempo.update(knee_raw)

        # ── 배경 레이어 ──
        if 'depth' in self._on:
            hy = (lms[23].y + lms[24].y) / 2
            self.depth_line.draw(frame, hy)

        if 'ktg' in self._on:
            self.ktg.draw(frame, lms)

        # ── 스켈레톤 레이어 ──
        if 'heatmap' in self._on:
            self.heatmap.draw_skeleton(
                frame, lms, deltas, metric_to_lm,
                connections, threshold=15.0)

        # ── 관절 게이지 ──
        if 'arc' in self._on:
            lknee_x = self.pf.fv('arc_lkx', lms[25].x) * W
            lknee_y = self.pf.fv('arc_lky', lms[25].y) * H
            self.arc.draw(
                frame,
                joint_px=(lknee_x, lknee_y),
                user_angle_raw=knee_raw,
                expert_angle=expert_metrics.get('knee_angle', 170.0),
                key='lknee', radius=26)

        # ── 사이드 UI ──
        if 'rom' in self._on:
            self.rom.draw(frame, knee_raw,
                          expert_metrics.get('knee_angle', None))

        if 'score_bar' in self._on:
            self.score_bar.draw(frame, deltas, exercise,
                                x=W-130, y=8, bar_w=110)

        if 'phase' in self._on:
            self.phase.draw(frame, self._cur_phase,
                            x=W-68, y=30)

        if 'symmetry' in self._on:
            self.symmetry.draw(frame, lms,
                               x=8, y=290, bar_w=90)

        if 'tempo' in self._on:
            self.tempo.draw(frame, x=W//2-60, y=10, w=120, h=28)

        if 'acl' in self._on:
            self.acl.draw(frame, user_metrics_raw, deltas,
                          x=14, y=H-90)

        if 'metricbar' in self._on:
            km = {'squat':   ['knee_angle','hip_angle','spine_lean','foot_width'],
                  'pushup':  ['elbow_angle','body_angle','spine_lean'],
                  'deadlift':['hip_angle','knee_angle','spine_lean']}
            lb = {'knee_angle':'무릎','hip_angle':'힙  ',
                  'spine_lean':'척추','foot_width':'발폭',
                  'elbow_angle':'팔꿈','body_angle':'몸통'}
            self.metricbar.draw_all(
                frame, user_metrics_raw, expert_metrics,
                km.get(exercise, ['knee_angle','hip_angle']),
                lb, x=W-185, y=215, bar_w=105)

        return frame
# ================================================================

def extract_expert_depth_target(ex_frames, exercise='squat'):
    """
    전문가 JSON에서 최저점 프레임의 힙 y좌표(0~1) 추출.
    DepthLine.set_expert_target()에 전달하여 사용.
    """
    if not ex_frames:
        return 0.7
    ak = {'squat':'knee_angle','deadlift':'hip_angle',
          'pushup':'elbow_angle'}.get(exercise,'knee_angle')
    best_frame, best_angle = None, float('inf')
    for fd in ex_frames:
        a = fd.get('metrics',{}).get(ak, 180.0)
        if a < best_angle:
            best_angle, best_frame = a, fd
    if best_frame is None:
        return 0.7
    lms = best_frame.get('landmarks', [])
    if len(lms) >= 25:
        return (lms[23][1] + lms[24][1]) / 2.0
    return 0.7

import types as _types
fo = _types.SimpleNamespace(FeedbackRenderer=FeedbackRenderer, extract_expert_depth_target=extract_expert_depth_target)
print('feedback_overlay 코드 셀 내장 완료: feedback_overlay.py import 없음')


feedback_overlay 코드 셀 내장 완료: feedback_overlay.py import 없음


In [7]:
# ── Cell 4C: ST-GCN 실시간 추론 코드 내장 — model.realtime_stgcn_infer import 없음 ──
"""Realtime ST-GCN inference adapter.

Side-effect-free runtime adapter for the multitask ST-GCN checkpoints trained by
the training script.  This module intentionally does *not* import that
training script because it performs label/cache work at import time.
"""
from __future__ import annotations

import os
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional, Sequence

try:  # Keep basic import/count helpers usable in minimal shells.
    import numpy as np  # type: ignore
    _NUMPY_IMPORT_ERROR: Optional[BaseException] = None
except Exception as exc:  # pragma: no cover - depends on local environment
    np = None  # type: ignore
    _NUMPY_IMPORT_ERROR = exc

try:  # Caught import: default-shell environments may not have torch installed.
    import torch  # type: ignore
    import torch.nn as nn  # type: ignore
    import torch.nn.functional as F  # type: ignore
    _TORCH_IMPORT_ERROR: Optional[BaseException] = None
except Exception as exc:  # pragma: no cover - depends on local environment
    torch = None  # type: ignore
    nn = None  # type: ignore
    F = None  # type: ignore
    _TORCH_IMPORT_ERROR = exc


CLASS_LIST = ["squat", "benchpress", "deadlift"]
PHASE_NAMES = ["ready", "down", "up"]
PHASE_READY = 0
PHASE_DOWN = 1
PHASE_UP = 2
NUM_KPT = 33
NUM_CLASSES = len(CLASS_LIST)
NUM_PHASES = len(PHASE_NAMES)

REPO_ROOT = Path(BASE).resolve() if 'BASE' in globals() else Path.cwd().resolve()
DEFAULT_BEST_CHECKPOINT = (
    REPO_ROOT
    / "phase_experiments"
    / "temporal_head_full_ablation"
    / "mlp_h128_c16_pw_2.0_ts2_do03_aug1"
    / "best_ep013.pt"
)
DEFAULT_LATEST_CHECKPOINT = DEFAULT_BEST_CHECKPOINT.with_name("latest.pt")



class RealtimeModelUnavailable(RuntimeError):
    """Raised when the realtime model cannot be used safely."""


def _require_numpy() -> None:
    if np is None:
        raise RealtimeModelUnavailable(f"numpy unavailable: {_NUMPY_IMPORT_ERROR}")


@dataclass
class InferenceState:
    available: bool = False
    ready: bool = False
    pred_class: Optional[str] = None
    pred_class_id: Optional[int] = None
    action_confidence: float = 0.0
    action_probs: Optional[np.ndarray] = None
    phase: str = "ready"
    phase_id: int = PHASE_READY
    phase_confidence: float = 0.0
    phase_probs: Optional[np.ndarray] = None
    count: int = 0
    last_increment: bool = False
    checkpoint_path: Optional[str] = None
    clip_len: Optional[int] = None
    warning: Optional[str] = None
    unavailable_reason: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "available": self.available,
            "ready": self.ready,
            "pred_class": self.pred_class,
            "pred_class_id": self.pred_class_id,
            "action_confidence": self.action_confidence,
            "action_probs": None if self.action_probs is None else self.action_probs.tolist(),
            "phase": self.phase,
            "phase_id": self.phase_id,
            "phase_confidence": self.phase_confidence,
            "phase_probs": None if self.phase_probs is None else self.phase_probs.tolist(),
            "count": self.count,
            "last_increment": self.last_increment,
            "checkpoint_path": self.checkpoint_path,
            "clip_len": self.clip_len,
            "warning": self.warning,
            "unavailable_reason": self.unavailable_reason,
        }


def build_adjacency(num_node: int = 33, edges: Optional[Sequence[tuple[int, int]]] = None) -> np.ndarray:
    _require_numpy()
    edge_list = MP_EDGES if edges is None else edges
    adj = np.zeros((num_node, num_node), dtype=np.float32)
    for i, j in edge_list:
        adj[i, j] = adj[j, i] = 1.0
    adj += np.eye(num_node, dtype=np.float32)
    degree = adj.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-6))
    return inv_sqrt @ adj @ inv_sqrt


# MediaPipe pose graph edges, matching the training-time graph.
MP_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12),
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

_A_NORM_NP = build_adjacency() if np is not None else None
A_NORM = torch.from_numpy(_A_NORM_NP).float() if torch is not None and _A_NORM_NP is not None else None


if nn is not None:

    class GraphConv(nn.Module):  # type: ignore[misc]
        def __init__(self, in_c: int, out_c: int, A: Any):
            super().__init__()
            self.register_buffer("A", A)
            self.conv = nn.Conv2d(in_c, out_c, 1)

        def forward(self, x: Any) -> Any:
            return torch.einsum("nctv,vw->nctw", self.conv(x), self.A)


    class STGCNBlock(nn.Module):  # type: ignore[misc]
        def __init__(self, in_c: int, out_c: int, A: Any, kernel_t: int = 9,
                     stride: int = 1, residual: bool = True):
            super().__init__()
            pad = (kernel_t - 1) // 2
            self.gcn = GraphConv(in_c, out_c, A)
            self.tcn = nn.Sequential(
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, (kernel_t, 1), stride=(stride, 1), padding=(pad, 0)),
                nn.BatchNorm2d(out_c),
            )
            if not residual:
                self.residual = lambda _: 0
            elif in_c == out_c and stride == 1:
                self.residual = nn.Identity()
            else:
                self.residual = nn.Sequential(
                    nn.Conv2d(in_c, out_c, 1, stride=(stride, 1)),
                    nn.BatchNorm2d(out_c),
                )
            self.relu = nn.ReLU(inplace=True)

        def forward(self, x: Any) -> Any:
            res = self.residual(x)
            x = self.tcn(self.gcn(x))
            return self.relu(x + res)


    class STGCNBackbone(nn.Module):  # type: ignore[misc]
        def __init__(self, in_channels: int = 3, A: Any = None, base: int = 64):
            super().__init__()
            graph = A_NORM if A is None else A
            if graph is None:
                raise RealtimeModelUnavailable("torch unavailable: cannot create graph tensor")
            self.data_bn = nn.BatchNorm1d(in_channels * graph.size(0))
            self.layers = nn.ModuleList([
                STGCNBlock(in_channels, base, graph, residual=False),
                STGCNBlock(base, base, graph),
                STGCNBlock(base, base, graph),
                STGCNBlock(base, base * 2, graph, stride=2),
                STGCNBlock(base * 2, base * 2, graph),
                STGCNBlock(base * 2, base * 4, graph, stride=2),
                STGCNBlock(base * 4, base * 4, graph),
            ])
            self.out_channels = base * 4

        def forward(self, x: Any) -> Any:
            n, c, t, v, m = x.size()
            x = x.permute(0, 4, 3, 1, 2).contiguous().view(n * m, v * c, t)
            x = self.data_bn(x)
            x = x.view(n, m, v, c, t).permute(0, 1, 3, 4, 2).contiguous().view(n * m, c, t, v)
            for block in self.layers:
                x = block(x)
            cout, tout, vout = x.size(1), x.size(2), x.size(3)
            return x.view(n, m, cout, tout, vout).mean(dim=1)


    class MultiTaskSTGCNLSTM(nn.Module):  # type: ignore[misc]
        def __init__(self, num_action: int = NUM_CLASSES, num_phase: int = NUM_PHASES,
                     in_c: int = 3, lstm_hidden: int = 128, lstm_layers: int = 1,
                     dropout: float = 0.3):
            super().__init__()
            self.backbone = STGCNBackbone(in_channels=in_c)
            channels = self.backbone.out_channels
            self.action_head = nn.Sequential(
                nn.Linear(channels, channels // 2), nn.ReLU(inplace=True),
                nn.Dropout(dropout), nn.Linear(channels // 2, num_action),
            )
            self.lstm = nn.LSTM(
                input_size=channels,
                hidden_size=lstm_hidden,
                num_layers=lstm_layers,
                batch_first=True,
                dropout=dropout if lstm_layers > 1 else 0.0,
                bidirectional=False,
            )
            self.phase_head = nn.Sequential(
                nn.Linear(lstm_hidden, lstm_hidden), nn.ReLU(inplace=True),
                nn.Dropout(dropout), nn.Linear(lstm_hidden, num_phase),
            )

        def forward(self, x: Any) -> Any:
            feat = self.backbone(x)
            action_logit = self.action_head(feat.mean(dim=(2, 3)))
            temporal = feat.mean(dim=-1).permute(0, 2, 1)
            lstm_out, _ = self.lstm(temporal)
            phase_logit = self.phase_head(lstm_out[:, -1, :])
            return action_logit, phase_logit


    class MultiTaskSTGCNMLP(nn.Module):  # type: ignore[misc]
        def __init__(self, num_action: int = NUM_CLASSES, num_phase: int = NUM_PHASES,
                     in_c: int = 3, mlp_hidden: int = 128, dropout: float = 0.3):
            super().__init__()
            self.backbone = STGCNBackbone(in_channels=in_c)
            channels = self.backbone.out_channels
            self.action_head = nn.Sequential(
                nn.Linear(channels, channels // 2), nn.ReLU(inplace=True),
                nn.Dropout(dropout), nn.Linear(channels // 2, num_action),
            )
            self.phase_head = nn.Sequential(
                nn.Linear(channels, mlp_hidden), nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(mlp_hidden, mlp_hidden // 2), nn.ReLU(inplace=True),
                nn.Dropout(dropout), nn.Linear(mlp_hidden // 2, num_phase),
            )

        def forward(self, x: Any) -> Any:
            feat = self.backbone(x)
            pooled = feat.mean(dim=(2, 3))
            return self.action_head(pooled), self.phase_head(pooled)

else:

    class _TorchUnavailablePlaceholder:
        def __init__(self, *_: Any, **__: Any):
            raise RealtimeModelUnavailable(f"torch unavailable: {_TORCH_IMPORT_ERROR}")

    class GraphConv(_TorchUnavailablePlaceholder):
        pass

    class STGCNBlock(_TorchUnavailablePlaceholder):
        pass

    class STGCNBackbone(_TorchUnavailablePlaceholder):
        pass

    class MultiTaskSTGCNLSTM(_TorchUnavailablePlaceholder):
        pass

    class MultiTaskSTGCNMLP(_TorchUnavailablePlaceholder):
        pass


def normalize_kpts(k: np.ndarray) -> np.ndarray:
    """Normalize keypoints with the same scale/centering used during training."""
    _require_numpy()
    arr = np.asarray(k, dtype=np.float32)
    if arr.ndim != 3 or arr.shape[1:] != (NUM_KPT, 3):
        raise ValueError(f"expected keypoints shape [T, 33, 3], got {arr.shape}")
    xy = arr[..., :2].copy()
    conf = arr[..., 2:3].copy()
    hip = (xy[:, 23] + xy[:, 24]) / 2
    sho = (xy[:, 11] + xy[:, 12]) / 2
    scale = np.linalg.norm(sho - hip, axis=1, keepdims=True) + 1e-6
    xy = (xy - hip[:, None, :]) / scale[:, None, :]
    return np.concatenate([xy, conf], axis=-1).astype(np.float32)


def landmarks_to_kpts(lms: Iterable[Any]) -> np.ndarray:
    """Convert MediaPipe landmark objects to [33, 3] = [x, y, visibility]."""
    _require_numpy()
    if lms is None:
        raise ValueError("landmarks are None")
    rows: list[list[float]] = []
    for lm in list(lms)[:NUM_KPT]:
        rows.append([
            float(getattr(lm, "x", 0.0)),
            float(getattr(lm, "y", 0.0)),
            float(getattr(lm, "visibility", 1.0)),
        ])
    while len(rows) < NUM_KPT:
        rows.append([0.0, 0.0, 0.0])
    return np.asarray(rows, dtype=np.float32)


def resolve_checkpoint_path(checkpoint_path: Optional[str | os.PathLike[str]] = None,
                            repo_root: Optional[str | os.PathLike[str]] = None) -> tuple[Optional[Path], Optional[str]]:
    root = Path(repo_root).resolve() if repo_root is not None else REPO_ROOT
    best = root / DEFAULT_BEST_CHECKPOINT.relative_to(REPO_ROOT)
    latest = root / DEFAULT_LATEST_CHECKPOINT.relative_to(REPO_ROOT)

    if checkpoint_path:
        return Path(checkpoint_path).expanduser(), None

    for env_name in ("TRAINED_MODEL_CHECKPOINT", "REALTIME_STGCN_CHECKPOINT"):
        env_path = os.environ.get(env_name)
        if env_path:
            return Path(env_path).expanduser(), None

    if best.exists():
        return best, None
    if latest.exists():
        return latest, "latest.pt selected because best checkpoint was not found; latest may not be deployable-best"
    return None, "no realtime ST-GCN checkpoint found"


def _require_torch() -> None:
    if torch is None or nn is None or F is None:
        raise RealtimeModelUnavailable(f"torch unavailable: {_TORCH_IMPORT_ERROR}")


def _torch_load(path: Path, map_location: Any) -> Any:
    _require_torch()
    try:
        return torch.load(path, map_location=map_location, weights_only=False)  # type: ignore[union-attr]
    except TypeError:
        return torch.load(path, map_location=map_location)  # type: ignore[union-attr]


def _state_dict_from_checkpoint(ckpt: Any) -> Mapping[str, Any]:
    if isinstance(ckpt, Mapping):
        for key in ("model", "model_state", "state_dict"):
            val = ckpt.get(key)
            if isinstance(val, Mapping):
                return val
        # Raw state_dict fallback; metadata validation elsewhere decides if allowed.
        if ckpt and all(hasattr(v, "shape") for v in ckpt.values()):
            return ckpt
    raise RealtimeModelUnavailable("checkpoint does not contain a recognized model state_dict")


def _strip_module_prefix(state: Mapping[str, Any]) -> dict[str, Any]:
    if not any(str(k).startswith("module.") for k in state.keys()):
        return dict(state)
    return {str(k).removeprefix("module."): v for k, v in state.items()}


def _metadata_from_checkpoint(ckpt: Any,
                              trusted_metadata: Optional[Mapping[str, Any]] = None) -> dict[str, Any]:
    trusted = dict(trusted_metadata or {})
    data = dict(ckpt) if isinstance(ckpt, Mapping) else {}
    cfg = data.get("cfg") if isinstance(data.get("cfg"), Mapping) else {}
    metadata = {
        "classes": data.get("classes", trusted.get("classes")),
        "phase_names": data.get("phase_names", trusted.get("phase_names")),
        "cfg": {**dict(trusted.get("cfg", {})), **dict(cfg)},
        "clip_len": data.get("clip_len", trusted.get("clip_len")),
    }
    if metadata["clip_len"] is None:
        metadata["clip_len"] = metadata["cfg"].get("clip_len")
    return metadata


def validate_checkpoint_metadata(ckpt: Any,
                                 clip_len_override: Optional[int] = None,
                                 trusted_metadata: Optional[Mapping[str, Any]] = None) -> dict[str, Any]:
    metadata = _metadata_from_checkpoint(ckpt, trusted_metadata=trusted_metadata)
    classes = metadata.get("classes")
    phase_names = metadata.get("phase_names")
    if list(classes or []) != CLASS_LIST:
        raise RealtimeModelUnavailable(
            f"checkpoint class order missing/mismatch: expected {CLASS_LIST}, got {classes!r}"
        )
    if list(phase_names or []) != PHASE_NAMES:
        raise RealtimeModelUnavailable(
            f"checkpoint phase order missing/mismatch: expected {PHASE_NAMES}, got {phase_names!r}"
        )

    cfg = dict(metadata.get("cfg") or {})
    model_type = str(cfg.get("model_type", "")).lower()
    if model_type not in {"mlp", "lstm"}:
        raise RealtimeModelUnavailable(f"unsupported/missing model_type in checkpoint cfg: {model_type!r}")
    if "hidden" not in cfg:
        raise RealtimeModelUnavailable("checkpoint cfg missing hidden")
    if "dropout" not in cfg:
        raise RealtimeModelUnavailable("checkpoint cfg missing dropout")
    if model_type == "lstm" and "lstm_layers" not in cfg:
        raise RealtimeModelUnavailable("checkpoint cfg missing lstm_layers")

    clip_len = clip_len_override if clip_len_override is not None else metadata.get("clip_len")
    if clip_len is None:
        raise RealtimeModelUnavailable("checkpoint missing clip_len and no override was provided")
    cfg["clip_len"] = int(clip_len)
    cfg["hidden"] = int(cfg["hidden"])
    cfg["dropout"] = float(cfg["dropout"])
    cfg["lstm_layers"] = int(cfg.get("lstm_layers", 1))
    cfg["model_type"] = model_type
    return cfg


def build_model_from_cfg(cfg: Mapping[str, Any], device: Any = "cpu") -> Any:
    _require_torch()
    model_type = str(cfg["model_type"]).lower()
    if model_type == "lstm":
        model = MultiTaskSTGCNLSTM(
            num_action=NUM_CLASSES,
            num_phase=NUM_PHASES,
            in_c=3,
            lstm_hidden=int(cfg["hidden"]),
            lstm_layers=int(cfg.get("lstm_layers", 1)),
            dropout=float(cfg["dropout"]),
        )
    else:
        model = MultiTaskSTGCNMLP(
            num_action=NUM_CLASSES,
            num_phase=NUM_PHASES,
            in_c=3,
            mlp_hidden=int(cfg["hidden"]),
            dropout=float(cfg["dropout"]),
        )
    return model.to(device)


@dataclass
class PhaseCounterState:
    phase: str
    phase_id: int
    count: int
    last_increment: bool
    segment_len: int


class OnlinePhaseCounter:
    """Streaming counterpart of count_phases(): count when valid UP segment ends."""

    def __init__(self, smooth_window: int = 1, min_up_len: int = 3,
                 require_prior_down: bool = False):
        self.smooth_window = max(1, int(smooth_window))
        self.min_up_len = max(1, int(min_up_len))
        self.require_prior_down = bool(require_prior_down)
        self._window: deque[int] = deque(maxlen=self.smooth_window)
        self.reset()

    def reset(self) -> None:
        self._window.clear()
        self.count = 0
        self.last_increment = False
        self.current_phase_id: Optional[int] = None
        self.segment_len = 0
        self._seen_down_since_count = False
        self._up_started_after_down = False

    def clear_frame(self) -> PhaseCounterState:
        self.last_increment = False
        phase_id = self.current_phase_id if self.current_phase_id is not None else PHASE_READY
        return PhaseCounterState(
            phase=PHASE_NAMES[phase_id],
            phase_id=phase_id,
            count=self.count,
            last_increment=False,
            segment_len=self.segment_len,
        )

    def _phase_to_id(self, phase: int | str) -> int:
        if isinstance(phase, str):
            if phase not in PHASE_NAMES:
                raise ValueError(f"unknown phase name: {phase!r}")
            return PHASE_NAMES.index(phase)
        idx = int(phase)
        if idx < 0 or idx >= NUM_PHASES:
            raise ValueError(f"unknown phase id: {idx}")
        return idx

    def _smooth(self, phase_id: int) -> int:
        self._window.append(phase_id)
        counts: dict[int, int] = {}
        for value in self._window:
            counts[value] = counts.get(value, 0) + 1
        latest_index = {value: idx for idx, value in enumerate(self._window)}
        return max(counts, key=lambda value: (counts[value], latest_index[value]))

    def update(self, phase: int | str) -> PhaseCounterState:
        self.last_increment = False
        phase_id = self._smooth(self._phase_to_id(phase))

        if self.current_phase_id is None:
            self.current_phase_id = phase_id
            self.segment_len = 1
            if phase_id == PHASE_DOWN:
                self._seen_down_since_count = True
            if phase_id == PHASE_UP:
                self._up_started_after_down = self._seen_down_since_count
            return self._state(phase_id)

        if phase_id == self.current_phase_id:
            self.segment_len += 1
            if phase_id == PHASE_DOWN:
                self._seen_down_since_count = True
            return self._state(phase_id)

        old_phase = self.current_phase_id
        old_len = self.segment_len
        up_is_valid = old_phase == PHASE_UP and old_len >= self.min_up_len
        prior_down_ok = (not self.require_prior_down) or self._up_started_after_down
        if up_is_valid and prior_down_ok:
            self.count += 1
            self.last_increment = True
            self._seen_down_since_count = False

        self.current_phase_id = phase_id
        self.segment_len = 1
        if phase_id == PHASE_DOWN:
            self._seen_down_since_count = True
        if phase_id == PHASE_UP:
            self._up_started_after_down = self._seen_down_since_count
        elif phase_id != PHASE_UP:
            self._up_started_after_down = False

        return self._state(phase_id)

    def _state(self, phase_id: int) -> PhaseCounterState:
        return PhaseCounterState(
            phase=PHASE_NAMES[phase_id],
            phase_id=phase_id,
            count=self.count,
            last_increment=self.last_increment,
            segment_len=self.segment_len,
        )


class RealtimeSTGCNInfer:
    """Online inference wrapper for multitask ST-GCN action + phase/count."""

    def __init__(self,
                 checkpoint_path: Optional[str | os.PathLike[str]] = None,
                 device: str = "auto",
                 clip_len_override: Optional[int] = None,
                 smooth_window: int = 5,
                 min_up_len: int = 3,
                 inference_interval: int = 1,
                 require_prior_down: bool = False,
                 required: bool = False,
                 trusted_metadata: Optional[Mapping[str, Any]] = None,
                 repo_root: Optional[str | os.PathLike[str]] = None):
        self.required = bool(required)
        self.available = False
        self.unavailable_reason: Optional[str] = None
        self.warning: Optional[str] = None
        self.device = "cpu"
        self.model = None
        self.cfg: dict[str, Any] = {}
        self.clip_len: Optional[int] = None
        self.checkpoint_path: Optional[Path] = None
        self.inference_interval = max(1, int(inference_interval))
        self._frame_index = 0
        self._buffer: deque[np.ndarray] = deque()
        self.counter = OnlinePhaseCounter(
            smooth_window=smooth_window,
            min_up_len=min_up_len,
            require_prior_down=require_prior_down,
        )
        self.state = InferenceState(
            available=False,
            unavailable_reason="not initialized",
            warning=None,
        )
        try:
            self._initialize(
                checkpoint_path=checkpoint_path,
                device=device,
                clip_len_override=clip_len_override,
                trusted_metadata=trusted_metadata,
                repo_root=repo_root,
            )
        except RealtimeModelUnavailable as exc:
            self._mark_unavailable(str(exc))
        except Exception as exc:  # pragma: no cover - defensive runtime fallback
            self._mark_unavailable(f"unexpected realtime model init failure: {exc}")

        if self.required and not self.available:
            raise RealtimeModelUnavailable(self.unavailable_reason or "realtime model unavailable")

    def _initialize(self,
                    checkpoint_path: Optional[str | os.PathLike[str]],
                    device: str,
                    clip_len_override: Optional[int],
                    trusted_metadata: Optional[Mapping[str, Any]],
                    repo_root: Optional[str | os.PathLike[str]]) -> None:
        _require_numpy()
        _require_torch()
        selected, warning = resolve_checkpoint_path(checkpoint_path, repo_root=repo_root)
        if selected is None:
            raise RealtimeModelUnavailable(warning or "checkpoint not found")
        if not selected.exists():
            raise RealtimeModelUnavailable(f"checkpoint not found: {selected}")

        resolved_device = self._resolve_device(device)
        ckpt = _torch_load(selected, map_location=resolved_device)
        cfg = validate_checkpoint_metadata(
            ckpt,
            clip_len_override=clip_len_override,
            trusted_metadata=trusted_metadata,
        )
        model = build_model_from_cfg(cfg, device=resolved_device)
        state_dict = _strip_module_prefix(_state_dict_from_checkpoint(ckpt))
        model.load_state_dict(state_dict, strict=True)
        model.eval()

        self.available = True
        self.unavailable_reason = None
        self.warning = warning
        self.device = str(resolved_device)
        self.model = model
        self.cfg = cfg
        self.clip_len = int(cfg["clip_len"])
        self.checkpoint_path = selected
        self._buffer = deque(maxlen=self.clip_len)
        self.state = InferenceState(
            available=True,
            ready=False,
            checkpoint_path=str(selected),
            clip_len=self.clip_len,
            warning=warning,
        )

    def _resolve_device(self, device: str) -> Any:
        _require_torch()
        if device == "auto":
            return "cuda" if torch.cuda.is_available() else "cpu"  # type: ignore[union-attr]
        if device == "cuda" and not torch.cuda.is_available():  # type: ignore[union-attr]
            return "cpu"
        return device

    def _mark_unavailable(self, reason: str) -> None:
        self.available = False
        self.unavailable_reason = reason
        self.state = InferenceState(
            available=False,
            ready=False,
            count=self.counter.count,
            last_increment=False,
            unavailable_reason=reason,
            checkpoint_path=None if self.checkpoint_path is None else str(self.checkpoint_path),
            clip_len=self.clip_len,
            warning=self.warning,
        )

    def reset(self) -> None:
        self._frame_index = 0
        self._buffer.clear()
        self.counter.reset()
        self.state = InferenceState(
            available=self.available,
            ready=False,
            checkpoint_path=None if self.checkpoint_path is None else str(self.checkpoint_path),
            clip_len=self.clip_len,
            warning=self.warning,
            unavailable_reason=self.unavailable_reason,
        )

    def clear_frame(self) -> InferenceState:
        counter_state = self.counter.clear_frame()
        self.state.ready = False
        self.state.pred_class = None
        self.state.pred_class_id = None
        self.state.action_confidence = 0.0
        self.state.action_probs = None
        self.state.last_increment = False
        self.state.count = counter_state.count
        self.state.phase = counter_state.phase
        self.state.phase_id = counter_state.phase_id
        self.state.phase_confidence = 0.0
        self.state.phase_probs = None
        return self.state

    def update(self, lms: Iterable[Any]) -> InferenceState:
        return self.update_kpts(landmarks_to_kpts(lms))

    def update_kpts(self, kpts: np.ndarray) -> InferenceState:
        if not self.available or self.model is None or self.clip_len is None:
            self.counter.clear_frame()
            return self.state

        arr = np.asarray(kpts, dtype=np.float32)
        if arr.shape != (NUM_KPT, 3):
            raise ValueError(f"expected one frame [33, 3], got {arr.shape}")
        self._frame_index += 1
        self._buffer.append(arr)

        if len(self._buffer) < self.clip_len:
            self.state.ready = False
            self.state.last_increment = False
            self.state.count = self.counter.count
            return self.state

        if self._frame_index % self.inference_interval != 0:
            return self.clear_frame()

        clip = np.stack(list(self._buffer), axis=0)
        norm = normalize_kpts(clip)
        x = torch.from_numpy(norm).permute(2, 0, 1).unsqueeze(0).unsqueeze(-1).contiguous()  # type: ignore[union-attr]
        x = x.to(self.device)
        with torch.no_grad():  # type: ignore[union-attr]
            action_logit, phase_logit = self.model(x)
            action_prob = F.softmax(action_logit, dim=1).detach().cpu().numpy()[0]
            phase_prob = F.softmax(phase_logit, dim=1).detach().cpu().numpy()[0]

        pred_cls_id = int(action_prob.argmax())
        phase_id = int(phase_prob.argmax())
        counter_state = self.counter.update(phase_id)
        self.state = InferenceState(
            available=True,
            ready=True,
            pred_class=CLASS_LIST[pred_cls_id],
            pred_class_id=pred_cls_id,
            action_confidence=float(action_prob[pred_cls_id]),
            action_probs=action_prob.astype(np.float32),
            phase=counter_state.phase,
            phase_id=counter_state.phase_id,
            phase_confidence=float(phase_prob[phase_id]),
            phase_probs=phase_prob.astype(np.float32),
            count=counter_state.count,
            last_increment=counter_state.last_increment,
            checkpoint_path=None if self.checkpoint_path is None else str(self.checkpoint_path),
            clip_len=self.clip_len,
            warning=self.warning,
            unavailable_reason=None,
        )
        return self.state


def model_status_line(state: Optional[InferenceState]) -> str:
    if state is None:
        return "AI model: inactive"
    if not state.available:
        return f"AI model fallback: {state.unavailable_reason or 'unavailable'}"
    cls = state.pred_class or "warming"
    phase = state.phase or "ready"
    return (
        f"AI {cls} {state.action_confidence:.2f} | "
        f"phase {phase} {state.phase_confidence:.2f} | count {state.count}"
    )

print('ST-GCN 추론 코드 셀 내장 완료: model.realtime_stgcn_infer import 없음')
print(f'  기본 checkpoint = {DEFAULT_BEST_CHECKPOINT}')


ST-GCN 추론 코드 셀 내장 완료: model.realtime_stgcn_infer import 없음
  기본 checkpoint = C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\temporal_head_full_ablation\mlp_h128_c16_pw_2.0_ts2_do03_aug1\best_ep013.pt


In [8]:
# ── Cell 5: 공통지표 + fallback 기본지표 계산 함수 ───────────────
# 핵심 수정:
# 1) rt.compute_sideview_metrics 가 없어도 더 이상 터지지 않게 fallback 계산 추가
# 2) knee_angle / hip_angle / spine_lean 등 HUD에 필요한 기본 지표를 직접 계산
# 3) 사용자 landmark, 전문가 JSON landmark 모두 같은 함수로 처리

VIS_THR = getattr(rt, 'VISIBILITY_THRESHOLD', 0.4)


def get_xyv(lms, idx):
    """
    user lms(MediaPipe 객체 list) 또는 expert lms_raw(JSON list [[x,y,z,vis],...]) 양쪽 처리.
    반환: (np.array([x,y]), visibility)
    """
    p = lms[idx]
    if hasattr(p, 'x'):
        return (
            np.array([float(p.x), float(p.y)], dtype=np.float32),
            float(getattr(p, 'visibility', 1.0))
        )

    # JSON landmark: [x, y, z, visibility] 형태라고 가정
    x, y = float(p[0]), float(p[1])
    v    = float(p[3]) if len(p) > 3 else 1.0
    return np.array([x, y], dtype=np.float32), v


def midp(lms, a, b):
    """두 landmark 중점. 한쪽 visibility가 낮으면 더 잘 보이는 점 사용."""
    pa, va = get_xyv(lms, a)
    pb, vb = get_xyv(lms, b)

    if va >= VIS_THR and vb >= VIS_THR:
        return (pa + pb) / 2.0

    return pa if va >= vb else pb


def dist2d(a, b):
    return float(np.linalg.norm(a - b))


def angle_abc(a, b, c):
    """
    세 점 A-B-C가 만드는 각도.
    B가 꼭짓점.
    """
    ba = a - b
    bc = c - b
    n = np.linalg.norm(ba) * np.linalg.norm(bc)

    if n < 1e-9:
        return 0.0

    cosv = float(np.dot(ba, bc) / n)
    cosv = float(np.clip(cosv, -1.0, 1.0))
    return float(math.degrees(math.acos(cosv)))


def angle_to_vertical(a, b):
    """
    b -> a 벡터와 수직축 사이 각도.
    0도 = 수직에 가까움, 90도 = 수평에 가까움.
    """
    dx = float(a[0] - b[0])
    dy = float(a[1] - b[1])
    return abs(math.degrees(math.atan2(abs(dx), abs(dy) + 1e-9)))


def _vis_sum(lms, ids):
    total = 0.0
    for i in ids:
        try:
            _, v = get_xyv(lms, i)
            total += float(v)
        except Exception:
            pass
    return total


def _pick_visible_side(lms):
    """
    측면 영상에서는 한쪽 관절이 더 잘 잡히는 경우가 많으므로
    hip-knee-ankle visibility 합이 큰 쪽을 대표 side로 사용.
    """
    left_score  = _vis_sum(lms, [11, 13, 15, 23, 25, 27, 31])
    right_score = _vis_sum(lms, [12, 14, 16, 24, 26, 28, 32])

    if right_score > left_score:
        return {
            'shoulder': 12, 'elbow': 14, 'wrist': 16,
            'hip': 24, 'knee': 26, 'ankle': 28, 'foot': 32
        }

    return {
        'shoulder': 11, 'elbow': 13, 'wrist': 15,
        'hip': 23, 'knee': 25, 'ankle': 27, 'foot': 31
    }


def compute_base_sideview_metrics_local(lms, exercise='squat'):
    """
    realtime_compare_side.py 안의 compute_sideview_metrics가 없거나 실패할 때 쓰는 최소 fallback.
    HUD와 delta 계산에 필요한 핵심 지표를 직접 계산한다.
    """
    side = _pick_visible_side(lms)

    sh, _ = get_xyv(lms, side['shoulder'])
    el, _ = get_xyv(lms, side['elbow'])
    wr, _ = get_xyv(lms, side['wrist'])
    hp, _ = get_xyv(lms, side['hip'])
    kn, _ = get_xyv(lms, side['knee'])
    an, _ = get_xyv(lms, side['ankle'])
    ft, _ = get_xyv(lms, side['foot'])

    shoulder_c = midp(lms, 11, 12)
    hip_c      = midp(lms, 23, 24)
    torso_len  = max(dist2d(shoulder_c, hip_c), 1e-6)

    knee_angle  = angle_abc(hp, kn, an)
    hip_angle   = angle_abc(sh, hp, kn)
    ankle_angle = angle_abc(kn, an, ft)
    elbow_angle = angle_abc(sh, el, wr)

    spine_lean  = angle_to_vertical(shoulder_c, hip_c)
    trunk_lean  = spine_lean

    # hip center -> knee 벡터의 수직 기준 기울기. 기존 코드 호환용 proxy.
    hip_knee_angle = angle_to_vertical(kn, hip_c)

    # deadlift 계열 호환용 proxy
    hip_shoulder_y_ratio = abs(float(hip_c[1] - shoulder_c[1])) / torso_len

    # bench 계열 호환용 proxy
    wrist_elbow_x_ratio = abs(float(wr[0] - el[0])) / torso_len
    body_line_ratio = angle_to_vertical(shoulder_c, hip_c)

    return {
        'knee_angle':             float(knee_angle),
        'hip_angle':              float(hip_angle),
        'ankle_angle':            float(ankle_angle),
        'elbow_angle':            float(elbow_angle),
        'spine_lean':             float(spine_lean),
        'trunk_lean':             float(trunk_lean),
        'hip_knee_angle':         float(hip_knee_angle),
        'hip_shoulder_y_ratio':   float(hip_shoulder_y_ratio),
        'wrist_elbow_x_ratio':    float(wrist_elbow_x_ratio),
        'body_line_ratio':        float(body_line_ratio),
    }


def compute_common_metrics(lms, exercise='squat'):
    """
    공통지표 계산.
    모든 값은 2D normalized coordinate 기반 proxy.
    """
    nose,   _  = get_xyv(lms, 0)
    l_sh,   _  = get_xyv(lms, 11); r_sh,  _  = get_xyv(lms, 12)
    l_el,   _  = get_xyv(lms, 13); r_el,  _  = get_xyv(lms, 14)
    l_wr,   _  = get_xyv(lms, 15); r_wr,  _  = get_xyv(lms, 16)
    l_hip,  _  = get_xyv(lms, 23); r_hip, _  = get_xyv(lms, 24)
    l_kn,   _  = get_xyv(lms, 25); r_kn,  _  = get_xyv(lms, 26)
    l_an,   _  = get_xyv(lms, 27); r_an,  _  = get_xyv(lms, 28)
    l_heel, _  = get_xyv(lms, 29); r_heel,_  = get_xyv(lms, 30)
    l_foot, _  = get_xyv(lms, 31); r_foot,_  = get_xyv(lms, 32)

    shoulder_c = midp(lms, 11, 12)
    hip_c      = midp(lms, 23, 24)
    torso_len  = max(dist2d(shoulder_c, hip_c), 1e-6)

    left_foot_center  = (l_heel + l_foot) / 2.0
    right_foot_center = (r_heel + r_foot) / 2.0
    wrist_c           = (l_wr + r_wr) / 2.0

    foot_width             = dist2d(l_an, r_an) / torso_len
    foot_x_asymmetry       = abs(float(l_an[0] - r_an[0])) / torso_len
    foot_foreaft_asymmetry = abs(float(left_foot_center[0] - right_foot_center[0])) / torso_len

    left_foot_flatness     = abs(float(l_heel[1] - l_foot[1])) / torso_len
    right_foot_flatness    = abs(float(r_heel[1] - r_foot[1])) / torso_len
    foot_flatness          = max(left_foot_flatness, right_foot_flatness)

    hand_width           = dist2d(l_wr, r_wr) / torso_len
    hand_height_diff     = abs(float(l_wr[1] - r_wr[1])) / torso_len
    shoulder_height_diff = abs(float(l_sh[1] - r_sh[1])) / torso_len

    if exercise in ('squat', 'deadlift'):
        xs = [float(nose[0]), float(shoulder_c[0]), float(hip_c[0])]
        head_hip_line = (max(xs) - min(xs)) / torso_len
    else:
        ys = [float(nose[1]), float(shoulder_c[1]), float(hip_c[1])]
        head_hip_line = (max(ys) - min(ys)) / torso_len

    neck_lean             = angle_to_vertical(nose, shoulder_c)
    wrist_shoulder_y_diff = (float(wrist_c[1]) - float(shoulder_c[1])) / torso_len

    left_elbow_angle      = angle_abc(l_sh, l_el, l_wr)
    right_elbow_angle     = angle_abc(r_sh, r_el, r_wr)
    elbow_angle_avg       = (left_elbow_angle + right_elbow_angle) / 2.0

    trunk_lean = angle_to_vertical(shoulder_c, hip_c)

    return {
        'foot_width':              float(foot_width),
        'foot_x_asymmetry':        float(foot_x_asymmetry),
        'foot_foreaft_asymmetry':  float(foot_foreaft_asymmetry),
        'foot_flatness':           float(foot_flatness),
        'left_foot_flatness':      float(left_foot_flatness),
        'right_foot_flatness':     float(right_foot_flatness),
        'hand_width':              float(hand_width),
        'hand_height_diff':        float(hand_height_diff),
        'shoulder_height_diff':    float(shoulder_height_diff),
        'head_hip_line':           float(head_hip_line),
        'neck_lean':               float(neck_lean),
        'wrist_shoulder_y_diff':   float(wrist_shoulder_y_diff),
        'elbow_angle_avg':         float(elbow_angle_avg),
        'trunk_lean':              float(trunk_lean),
    }


def _call_rt_compute_sideview_metrics(lms, exercise='squat'):
    """
    rt.compute_sideview_metrics가 있으면 사용하되,
    없거나 signature가 달라도 notebook이 멈추지 않도록 처리.
    """
    fn = getattr(rt, 'compute_sideview_metrics', None)

    if not callable(fn):
        return {}

    for args in [(lms, exercise), (lms,)]:
        try:
            out = fn(*args)
            return dict(out) if isinstance(out, dict) else {}
        except TypeError:
            continue
        except Exception as e:
            print(f'[경고] rt.compute_sideview_metrics 실패 → fallback 사용: {e}')
            return {}

    return {}


def compute_user_metrics_plus(lms, exercise='squat'):
    """
    기존 rt 지표 + fallback 기본지표 + 공통지표 합산.
    rt.compute_sideview_metrics가 없어도 knee_angle 등이 반드시 나오게 만든다.
    """
    local_base = compute_base_sideview_metrics_local(lms, exercise)
    rt_base    = _call_rt_compute_sideview_metrics(lms, exercise)
    common     = compute_common_metrics(lms, exercise)

    merged = {}
    merged.update(local_base)
    merged.update(rt_base)   # rt에 같은 key가 있으면 rt 값을 우선 사용
    merged.update(common)    # 공통지표는 최종 반영
    return merged


def compute_expert_metrics_plus(ex_lms_raw, ex_metrics_json=None, exercise='squat'):
    """
    전문가 JSON 지표 + fallback 기본지표 + 공통지표 합산.
    expert JSON 내부 metrics가 비어 있어도 비교에 필요한 key를 만들어준다.
    """
    local_base = compute_base_sideview_metrics_local(ex_lms_raw, exercise)
    common     = compute_common_metrics(ex_lms_raw, exercise)

    json_base = {}
    if isinstance(ex_metrics_json, dict):
        json_base = dict(ex_metrics_json)

    merged = {}
    merged.update(local_base)
    merged.update(json_base)
    merged.update(common)
    return merged


print('공통지표 + fallback 기본지표 계산 함수 로드 완료')


공통지표 + fallback 기본지표 계산 함수 로드 완료


In [9]:
# ── Cell 6: 지표 설정 확장 ────────────────────────────────────────
# rt 모듈의 EXERCISE_METRICS / DELTA_THRESHOLDS / DELTA_TO_LM /
# DISPLAY_LABELS / METRIC_LABELS 를 공통지표 기반으로 확장.
# [실험적 기준] 표기 = heuristic threshold.

# ── compatibility patch: realtime_compare_side.py 구버전 보정 ──

if not hasattr(rt, 'DELTA_THRESHOLDS'):
    rt.DELTA_THRESHOLDS = {}

if not hasattr(rt, 'DELTA_TO_LM'):
    rt.DELTA_TO_LM = {}

if not hasattr(rt, 'DISPLAY_LABELS'):
    rt.DISPLAY_LABELS = {}

if not hasattr(rt, 'METRIC_LABELS'):
    rt.METRIC_LABELS = {}

if not hasattr(rt, 'EXERCISE_METRICS'):
    rt.EXERCISE_METRICS = {
        'squat': [],
        'deadlift': [],
        'bench': []
    }

print("구버전 realtime_compare_side.py 보정 완료")
ADDED_METRICS = {
    'squat': [
        'knee_angle', 'hip_angle', 'spine_lean', 'hip_knee_angle',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'elbow_angle_avg', 'trunk_lean',
    ],
    'deadlift': [
        'knee_angle', 'hip_angle', 'spine_lean', 'hip_shoulder_y_ratio',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'trunk_lean',
    ],
    'bench': [
        'elbow_angle', 'wrist_elbow_x_ratio', 'body_line_ratio',
        'foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry',
        'foot_flatness', 'hand_width', 'hand_height_diff',
        'shoulder_height_diff', 'head_hip_line', 'neck_lean',
        'wrist_shoulder_y_diff', 'knee_angle',
    ],
}
rt.EXERCISE_METRICS.update(ADDED_METRICS)

# ✅ 중요: 측면영상에서는 정면/45도 전용 지표를 오류판정에서 제외한다.
# 발너비, 양손거리, 양손높이차, 어깨높이차는 측면영상에서 겹침/가림 때문에
# delta가 튀기 쉽다. 이 값들이 오류판정에 들어가면 기존 지표가 전부 깨져 보인다.
SIDE_RELIABLE_METRICS = {
    'squat': [
        'knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean',
        'hip_knee_angle', 'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
    'deadlift': [
        'knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean',
        'hip_shoulder_y_ratio', 'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
    'bench': [
        'elbow_angle', 'wrist_elbow_x_ratio', 'body_line_ratio',
        'foot_flatness', 'head_hip_line', 'neck_lean'
    ],
}

FRONT_45_EXTRA_METRICS = {
    'squat': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
    'deadlift': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
    'bench': ['foot_width', 'foot_x_asymmetry', 'foot_foreaft_asymmetry', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'],
}

_view = str(globals().get('VIDEO_VIEW_MODE', 'side')).lower()
if _view in ['side', '측면']:
    rt.EXERCISE_METRICS[EXERCISE] = SIDE_RELIABLE_METRICS.get(EXERCISE, SIDE_RELIABLE_METRICS['squat'])
else:
    base = SIDE_RELIABLE_METRICS.get(EXERCISE, SIDE_RELIABLE_METRICS['squat'])
    extra = FRONT_45_EXTRA_METRICS.get(EXERCISE, [])
    rt.EXERCISE_METRICS[EXERCISE] = list(dict.fromkeys(base + extra))

# Delta threshold [실험적 기준] — 전문가와의 차이가 이 값을 넘으면 주의
rt.DELTA_THRESHOLDS.update({
    'foot_width':              0.15,
    'foot_x_asymmetry':        0.10,
    'foot_foreaft_asymmetry':  0.10,
    'foot_flatness':           0.04,
    'left_foot_flatness':      0.04,
    'right_foot_flatness':     0.04,
    'hand_width':              0.20,
    'hand_height_diff':        0.07,
    'shoulder_height_diff':    0.05,
    'head_hip_line':           0.10,
    'neck_lean':               10.0,
    'wrist_shoulder_y_diff':   0.15,
    'elbow_angle_avg':         12.0,
    'trunk_lean':              8.0,
})

# 관련 landmark 인덱스 (issue_marker 강조에 사용)
rt.DELTA_TO_LM.update({
    'foot_width':              [27, 28],
    'foot_x_asymmetry':        [27, 28],
    'foot_foreaft_asymmetry':  [27, 28, 29, 30, 31, 32],
    'foot_flatness':           [29, 30, 31, 32],
    'left_foot_flatness':      [29, 31],
    'right_foot_flatness':     [30, 32],
    'hand_width':              [15, 16],
    'hand_height_diff':        [15, 16],
    'shoulder_height_diff':    [11, 12],
    'head_hip_line':           [0, 11, 12, 23, 24],
    'neck_lean':               [0, 11, 12],
    'wrist_shoulder_y_diff':   [11, 12, 15, 16],
    'elbow_angle_avg':         [11, 12, 13, 14, 15, 16],
    'trunk_lean':              [11, 12, 23, 24],
})

DISPLAY_LABELS_KR = {
    'foot_width':              '발너비',
    'foot_x_asymmetry':        '발목X차',
    'foot_foreaft_asymmetry':  '발전후차',
    'foot_flatness':           '발바닥밀착',
    'left_foot_flatness':      '왼발밀착',
    'right_foot_flatness':     '오른발밀착',
    'hand_width':              '양손거리',
    'hand_height_diff':        '양손높이차',
    'shoulder_height_diff':    '어깨높이차',
    'head_hip_line':           '머리-골반정렬',
    'neck_lean':               '목머리각',
    'wrist_shoulder_y_diff':   '손목어깨높이',
    'elbow_angle_avg':         '팔꿈치각',
    'trunk_lean':              '몸통기울기',
    'knee_angle':              '무릎각',
    'hip_angle':               '엉덩이각',
    'spine_lean':              '척추기울기',
}
rt.DISPLAY_LABELS.update(DISPLAY_LABELS_KR)
rt.METRIC_LABELS.update(rt.DISPLAY_LABELS)

print('지표 설정 확장 완료')
print('현재 오류판정 사용 지표:', rt.EXERCISE_METRICS.get(EXERCISE, []))


구버전 realtime_compare_side.py 보정 완료
지표 설정 확장 완료
현재 오류판정 사용 지표: ['knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean', 'hip_knee_angle', 'foot_flatness', 'head_hip_line', 'neck_lean']


In [10]:
# ── Cell 7: 시각화 헬퍼 v4 ───────────────────────────────────────
# 목표:
# 1) 몸통 기울기 텍스트를 사람 몸 위에 작게 쓰지 않고, HUD에서 크게 설명
# 2) 한글 폰트가 없으면 네모(□□□)로 깨지지 않도록 영어 fallback 사용
# 3) 측면영상에서는 발너비를 오류판정/표시에서 제외
# 4) 기존 레이어와 새 레이어가 겹쳐서 지저분해지는 문제 완화

from PIL import Image as PilImage, ImageDraw, ImageFont
import glob
import os
import textwrap

# Skeleton 연결 정의
SKEL_CONNS = [
    (0,11),(0,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(24,26),(26,28),
    (27,29),(29,31),(28,30),(30,32),
]

# 색상 팔레트 (BGR)
C_EXP_BONE  = (80, 220, 120)
C_EXP_JOINT = (180, 255, 180)
C_BRACKET   = (0, 215, 255)
C_TRUNK_OK  = (80, 220, 100)
C_TRUNK_ERR = (60, 60, 255)
C_TRUNK_REF = (170, 170, 170)
C_ERR_CIR   = (0, 0, 255)

# ✅ 몸통 기울기 허용 오차
TRUNK_LEAN_TOL_DEG = 8.0

# 발너비 브라켓 표시 대상 뷰
FOOT_BRACKET_VIEW_MODES = {'front', '45deg', '45', 'diagonal', 'oblique', '정면', '45도', '사선'}


# ────────────────────────────────────────────────────────────────
# [한글 폰트 탐색]
# ────────────────────────────────────────────────────────────────
def find_korean_font():
    candidates = []

    # Cell 3에서 FONT_PATH를 지정했다면 먼저 확인
    try:
        if FONT_PATH:
            candidates.append(FONT_PATH)
    except NameError:
        pass

    # Colab / Ubuntu에서 자주 쓰는 경로들
    candidates += [
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
        '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]

    # 혹시 경로가 달라도 찾기
    patterns = [
        '/usr/share/fonts/**/*Nanum*.ttf',
        '/usr/share/fonts/**/*NotoSansCJK*.ttc',
        '/content/**/*Nanum*.ttf',
        '/content/**/*NotoSansCJK*.ttc',
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat, recursive=True))

    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None


KOREAN_FONT_PATH = find_korean_font()
FONT_KR_OK = KOREAN_FONT_PATH is not None

def _fnt(size):
    try:
        if FONT_KR_OK:
            return ImageFont.truetype(KOREAN_FONT_PATH, size)
    except Exception:
        pass
    return ImageFont.load_default()

def _ko(kr, en):
    """한글 폰트가 없으면 영어로 표시해서 네모 깨짐을 방지."""
    return kr if FONT_KR_OK else en


def draw_text_box_bgr(frame, lines, xy=(16, 16), font_size=20,
                      fg=(255,255,255), bg=(20,20,30), pad=8,
                      alpha=0.72):
    """
    BGR frame 위에 PIL로 읽기 쉬운 박스형 텍스트를 그림.
    lines: str 또는 list[str]
    fg/bg: RGB 기준
    """
    if isinstance(lines, str):
        lines = [lines]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    base = PilImage.fromarray(rgb).convert('RGBA')
    overlay = PilImage.new('RGBA', base.size, (0,0,0,0))
    drw = ImageDraw.Draw(overlay)

    font = _fnt(font_size)
    line_h = font_size + 6
    widths = []
    for line in lines:
        try:
            bbox = drw.textbbox((0,0), line, font=font)
            widths.append(bbox[2] - bbox[0])
        except Exception:
            widths.append(len(line) * font_size * 0.6)

    box_w = int(max(widths) + pad*2)
    box_h = int(len(lines)*line_h + pad*2)
    x, y = xy
    x = max(0, min(int(x), frame.shape[1] - box_w - 2))
    y = max(0, min(int(y), frame.shape[0] - box_h - 2))

    bg_rgba = (bg[0], bg[1], bg[2], int(255*alpha))
    try:
        drw.rounded_rectangle([x, y, x+box_w, y+box_h], radius=10, fill=bg_rgba)
    except Exception:
        drw.rectangle([x, y, x+box_w, y+box_h], fill=bg_rgba)

    for i, line in enumerate(lines):
        drw.text((x+pad, y+pad+i*line_h), line, font=font, fill=fg+(255,))

    composed = PilImage.alpha_composite(base, overlay).convert('RGB')
    return cv2.cvtColor(np.array(composed), cv2.COLOR_RGB2BGR)


# ────────────────────────────────────────────────────────────────
# [전문가 skeleton -> 별도 canvas]
# ────────────────────────────────────────────────────────────────
def draw_expert_on_canvas(canvas_w, canvas_h, ex_lms_raw, bad_indices=None):
    if bad_indices is None:
        bad_indices = set()

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    if ex_lms_raw is None or len(ex_lms_raw) < 33:
        cv2.putText(canvas, 'NO EXPERT DATA',
                    (canvas_w//2-80, canvas_h//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (80,80,80), 2)
        return canvas

    def to_px(idx):
        p = ex_lms_raw[idx]
        if hasattr(p, 'x'):
            x, y, v = float(p.x), float(p.y), float(getattr(p, 'visibility', 1.0))
        else:
            x, y = float(p[0]), float(p[1])
            v = float(p[3]) if len(p) > 3 else 1.0
        return (int(x * canvas_w), int(y * canvas_h)), v

    for (a, b) in SKEL_CONNS:
        pa, va = to_px(a)
        pb, vb = to_px(b)
        if va < VIS_THR or vb < VIS_THR:
            continue
        cv2.line(canvas, pa, pb, C_EXP_BONE, 2, cv2.LINE_AA)

    for idx in range(33):
        pt, v = to_px(idx)
        if v < VIS_THR:
            continue
        is_bad = idx in bad_indices
        r  = 6 if is_bad else 4
        c  = C_ERR_CIR if is_bad else C_EXP_JOINT
        cv2.circle(canvas, pt, r, c, -1, cv2.LINE_AA)
        if is_bad:
            cv2.circle(canvas, pt, r + 5, C_ERR_CIR, 2, cv2.LINE_AA)

    cv2.putText(canvas, 'EXPERT', (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, C_EXP_BONE, 2, cv2.LINE_AA)
    return canvas


# ── 발너비 bracket 표시 여부 판단 ────────────────────────────────
def infer_view_mode_from_pose(lms):
    """양발목 x거리/torso 비율로 정면/측면을 아주 거칠게 추정."""
    try:
        la, va = get_xyv(lms, 27)
        ra, vr = get_xyv(lms, 28)
        sc = midp(lms, 11, 12)
        hc = midp(lms, 23, 24)
        torso = max(dist2d(sc, hc), 1e-6)
        foot_x_gap = abs(float(la[0] - ra[0])) / torso
        if foot_x_gap > 0.45:
            return 'front'
        if foot_x_gap > 0.22:
            return '45deg'
        return 'side'
    except Exception:
        return 'side'


def should_draw_foot_bracket(lms=None, view_mode=None):
    vm = str(view_mode or globals().get('VIDEO_VIEW_MODE', 'side')).lower()
    if vm == 'auto':
        vm = infer_view_mode_from_pose(lms) if lms is not None else 'side'
    return vm in FOOT_BRACKET_VIEW_MODES


# ── 발너비 bracket 시각화 ────────────────────────────────────────
def draw_foot_bracket(frame, lms, metrics, view_mode=None):
    """
    정면/45도에서만 양발목 아래에 bracket 표시.
    측면영상에서는 발이 겹쳐 보이기 때문에 표시하지 않는다.
    """
    if not should_draw_foot_bracket(lms, view_mode=view_mode):
        return frame

    H, W = frame.shape[:2]
    la, va = get_xyv(lms, 27)
    ra, vr = get_xyv(lms, 28)
    if va < VIS_THR or vr < VIS_THR:
        return frame

    lx = int(la[0] * W);  ly = int(la[1] * H)
    rx = int(ra[0] * W);  ry = int(ra[1] * H)
    y   = min(H - 16, max(ly, ry) + 20)
    lx2 = min(lx, rx);  rx2 = max(lx, rx)

    if rx2 - lx2 < 12:
        return frame

    h = 8
    cv2.line(frame, (lx2, y),   (rx2, y),   C_BRACKET, 3)
    cv2.line(frame, (lx2, y-h), (lx2, y+h), C_BRACKET, 3)
    cv2.line(frame, (rx2, y-h), (rx2, y+h), C_BRACKET, 3)
    cv2.arrowedLine(frame, (lx2+18, y), (lx2, y), C_BRACKET, 2, tipLength=0.35)
    cv2.arrowedLine(frame, (rx2-18, y), (rx2, y), C_BRACKET, 2, tipLength=0.35)

    fw = metrics.get('foot_width')
    txt = _ko(f'발너비 {fw:.2f}' if fw is not None else '발너비',
              f'Foot width {fw:.2f}' if fw is not None else 'Foot width')
    frame = draw_text_box_bgr(frame, txt, (max(5, (lx2+rx2)//2 - 70), max(16, y - 50)),
                              font_size=16, fg=(255,230,80), bg=(10,10,20), alpha=0.65)
    return frame


# ── 몸통 기울기 부족/과도 판정 ───────────────────────────────────
def evaluate_trunk_lean_issue(user_metrics, expert_metrics=None, tol_deg=None):
    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)
    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None

    if u is None or not np.isfinite(u):
        return None

    if e is None or not np.isfinite(e):
        lo, hi = 5.0, 35.0
        if u < lo:
            return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 부족', 'Trunk lean too small'), 'landmarks': [11,12,23,24], 'delta': u - lo}
        if u > hi:
            return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 과도', 'Trunk lean too large'), 'landmarks': [11,12,23,24], 'delta': u - hi}
        return None

    diff = float(u) - float(e)
    if diff > tol:
        return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 과도', 'Trunk lean too large'), 'landmarks': [11,12,23,24], 'delta': diff}
    if diff < -tol:
        return {'key': 'trunk_lean', 'message': _ko('몸통 기울기 부족', 'Trunk lean too small'), 'landmarks': [11,12,23,24], 'delta': diff}
    return None


def trunk_status_text(user_metrics, expert_metrics=None, tol_deg=None):
    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None
    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)

    if u is None or not np.isfinite(u):
        return _ko('몸통: 측정불가', 'Trunk: N/A'), (150,150,150), None, e

    issue = evaluate_trunk_lean_issue(user_metrics, expert_metrics, tol)
    if issue is None:
        return _ko('몸통: 정상 범위', 'Trunk: OK'), (80,220,100), u, e
    return issue['message'], (80,80,255), u, e


def _point_from_vertical_angle(origin_px, length_px, angle_deg, sign):
    theta = math.radians(float(angle_deg))
    dx = sign * math.sin(theta) * length_px
    dy = -math.cos(theta) * length_px
    return (int(origin_px[0] + dx), int(origin_px[1] + dy))


# ── 몸통 기울기 zone 시각화 ──────────────────────────────────────
def draw_trunk_lean_zone(frame, lms, user_metrics, expert_metrics=None, bad_indices=None, tol_deg=None):
    """
    사람 몸 위에는 숫자 텍스트를 쓰지 않는다.
    - 초록/빨강 zone + 현재 몸통선만 표시
    - 자세한 숫자 설명은 오른쪽 HUD에서 크게 표시
    """
    if bad_indices is None:
        bad_indices = set()

    H, W = frame.shape[:2]
    sc_n = midp(lms, 11, 12)
    hc_n = midp(lms, 23, 24)
    sc   = (int(sc_n[0]*W), int(sc_n[1]*H))
    hc   = (int(hc_n[0]*W), int(hc_n[1]*H))

    torso_px = max(int(math.sqrt((sc[0]-hc[0])**2 + (sc[1]-hc[1])**2)), 40)
    guide_len = int(torso_px * 1.35)

    u = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    e = expert_metrics.get('trunk_lean') if isinstance(expert_metrics, dict) else None
    if u is None or not np.isfinite(u):
        return frame
    if e is None or not np.isfinite(e):
        e = u

    tol = float(tol_deg if tol_deg is not None else TRUNK_LEAN_TOL_DEG)
    lo  = max(0.0, float(e) - tol)
    hi  = min(85.0, float(e) + tol)

    issue = evaluate_trunk_lean_issue(user_metrics, expert_metrics, tol)
    is_bad = issue is not None
    color = C_TRUNK_ERR if is_bad else C_TRUNK_OK

    sign = 1 if (sc[0] - hc[0]) >= 0 else -1

    p_lo  = _point_from_vertical_angle(hc, guide_len, lo, sign)
    p_hi  = _point_from_vertical_angle(hc, guide_len, hi, sign)
    p_mid = _point_from_vertical_angle(hc, guide_len, float(e), sign)

    overlay = frame.copy()
    poly = np.array([hc, p_lo, p_hi], dtype=np.int32)
    cv2.fillPoly(overlay, [poly], color)
    frame = cv2.addWeighted(overlay, 0.14, frame, 0.86, 0)

    cv2.line(frame, hc, p_lo, color, 2, cv2.LINE_AA)
    cv2.line(frame, hc, p_hi, color, 2, cv2.LINE_AA)
    cv2.line(frame, hc, p_mid, C_TRUNK_REF, 1, cv2.LINE_AA)

    # 현재 사용자 몸통 중심선
    cv2.line(frame, hc, sc, color, 5, cv2.LINE_AA)
    cv2.circle(frame, hc, 6, color, -1, cv2.LINE_AA)
    cv2.circle(frame, sc, 6, color, -1, cv2.LINE_AA)
    return frame


# 이전 함수명 호환용 alias
def draw_trunk_lean_line(frame, lms, metrics, bad_indices=None):
    return draw_trunk_lean_zone(frame, lms, metrics, None, bad_indices)


# ── threshold 초과 관절에 빨간 원 ────────────────────────────────
def draw_issue_markers(frame, lms, deltas, max_items=3):
    """
    현재 view에서 실제 오류판정에 쓰는 지표만 표시한다.
    측면영상에서 foot_width 같은 정면용 지표가 빨간 원을 만드는 문제를 방지.
    """
    H, W = frame.shape[:2]
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    items = sorted(
        [
            (abs(d) / max(rt.DELTA_THRESHOLDS.get(k, 999), 1e-6), k, d)
            for k, d in deltas.items()
            if k in active
        ],
        reverse=True
    )[:max_items]

    for ratio, key, delta in items:
        if ratio < 1.0:
            continue
        for idx in rt.DELTA_TO_LM.get(key, []):
            p = lms[idx]
            v = float(getattr(p, 'visibility', 1.0))
            if v < VIS_THR:
                continue
            cx = int(p.x * W)
            cy = int(p.y * H)
            cv2.circle(frame, (cx, cy), 13, C_ERR_CIR, 2, cv2.LINE_AA)
            cv2.circle(frame, (cx, cy),  4, C_ERR_CIR, -1, cv2.LINE_AA)
    return frame


# ── 상단 피드백 배너 ─────────────────────────────────────────────
def draw_feedback_banner(frame, feedback_text, issue_key=None,
                         user_metrics=None, expert_metrics=None):
    if not feedback_text:
        feedback_text = _ko('정상 범위', 'OK')

    is_ok = any(w in str(feedback_text) for w in ['정상', 'OK', '좋', 'safe'])
    fg = (120, 245, 150) if is_ok else (255, 130, 130)

    # 몸통 상태는 HUD에서 충분히 설명하므로 배너는 한 줄만
    return draw_text_box_bgr(
        frame,
        str(feedback_text),
        xy=(16, 52),
        font_size=20,
        fg=fg,
        bg=(18, 18, 28),
        alpha=0.74
    )


# ── PIL 기반 HUD 패널 ────────────────────────────────────────────
HUD_METRICS_SIDE = [
    # (한글명, 영어명, metric key, unit)
    ('무릎각',        'Knee angle',      'knee_angle',     'deg'),
    ('엉덩이각',      'Hip angle',       'hip_angle',      'deg'),
    ('몸통기울기',    'Trunk lean',      'trunk_lean',     'deg'),
    ('머리-골반정렬', 'Head-hip line',   'head_hip_line',  ''),
    ('발바닥밀착',    'Foot flatness',   'foot_flatness',  ''),
]

HUD_METRICS_FRONT45 = HUD_METRICS_SIDE + [
    ('발너비',        'Foot width',      'foot_width',     ''),
    ('양손높이차',    'Hand height diff','hand_height_diff',''),
]

def _fmt_val(v, unit):
    if v is None or not isinstance(v, (int, float, np.floating)) or not np.isfinite(v):
        return 'N/A'
    return f'{float(v):.1f}°' if unit == 'deg' else f'{float(v):.2f}'


def make_hud_panel(panel_w, panel_h, user_metrics, deltas, count,
                   feedback_text, issue_key, fps, ex_idx, ex_total,
                   exercise='squat'):
    img  = PilImage.new('RGB', (panel_w, panel_h), (18, 18, 28))
    drw  = ImageDraw.Draw(img)

    fTITLE = _fnt(28)
    fBIG   = _fnt(24)
    fMD    = _fnt(17)
    fSM    = _fnt(14)

    y = 14
    EX = {'squat':'SQUAT','deadlift':'DEADLIFT','bench':'BENCH PRESS'}

    # 제목
    drw.text((14, y), EX.get(exercise, exercise.upper()), font=fTITLE, fill=(240, 205, 80))
    y += 42

    # 횟수
    drw.text((14, y), _ko('현재 횟수', 'Reps'), font=fMD, fill=(180,180,190))
    drw.text((panel_w-70, y-8), str(count), font=fTITLE, fill=(80, 220, 255))
    y += 42

    # 몸통 상태 요약
    trunk_msg, trunk_col_bgr, u_tr, e_tr = trunk_status_text(user_metrics, None, TRUNK_LEAN_TOL_DEG)
    # expert값은 deltas에서 복원
    if u_tr is None:
        u_tr = user_metrics.get('trunk_lean') if isinstance(user_metrics, dict) else None
    if u_tr is not None and 'trunk_lean' in deltas:
        e_tr = u_tr - deltas.get('trunk_lean', 0)

    trunk_col_rgb = (trunk_col_bgr[2], trunk_col_bgr[1], trunk_col_bgr[0])
    drw.rounded_rectangle([10, y, panel_w-10, y+68], radius=12,
                          fill=(34,34,48), outline=trunk_col_rgb, width=2)
    drw.text((22, y+8), trunk_msg, font=fBIG, fill=trunk_col_rgb)

    detail = f"U {_fmt_val(u_tr, 'deg')}  |  E {_fmt_val(e_tr, 'deg')}  |  ±{TRUNK_LEAN_TOL_DEG:.0f}°"
    drw.text((22, y+42), detail, font=fSM, fill=(205,205,215))
    y += 82

    # 진행 정보
    prog = _ko(f'전문가 프레임 {ex_idx+1}/{ex_total}   FPS {fps:.1f}',
               f'Expert {ex_idx+1}/{ex_total}   FPS {fps:.1f}')
    drw.text((14, y), prog, font=fSM, fill=(120,120,130))
    y += 25
    drw.line([(10,y),(panel_w-10,y)], fill=(60,60,80), width=1)
    y += 14

    # 지표 목록: 측면에서는 측면 신뢰 지표만 표시
    vm = str(globals().get('VIDEO_VIEW_MODE', 'side')).lower()
    metric_list = HUD_METRICS_FRONT45 if vm not in ['side', '측면'] else HUD_METRICS_SIDE
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    for kr_label, en_label, key, unit in metric_list:
        if key not in active and key not in ['knee_angle', 'hip_angle', 'trunk_lean']:
            continue
        if y > panel_h - 125:
            break

        label = _ko(kr_label, en_label)
        val = user_metrics.get(key) if isinstance(user_metrics, dict) else None
        delta = deltas.get(key) if isinstance(deltas, dict) else None
        thr = rt.DELTA_THRESHOLDS.get(key)

        if val is not None and delta is not None:
            ex_val = val - delta
        else:
            ex_val = None

        bad = False
        if delta is not None and thr is not None:
            bad = abs(float(delta)) > float(thr)

        col = (255,130,130) if bad else (120,245,150)
        drw.text((14, y), label, font=fMD, fill=(225,225,230))
        drw.text((panel_w-100, y), _ko('주의', 'WARN') if bad else 'OK',
                 font=fMD, fill=col)

        y += 22
        val_line = f"U {_fmt_val(val, unit)}  /  E {_fmt_val(ex_val, unit)}"
        if delta is not None:
            val_line += f"  Δ {_fmt_val(delta, unit)}"
        drw.text((22, y), val_line, font=fSM, fill=(170,170,180))

        # delta 기준 bar
        y += 20
        if delta is not None and thr is not None and float(thr) > 0:
            bw = panel_w - 44
            ratio = min(abs(float(delta)) / float(thr), 1.5) / 1.5
            drw.rectangle([22, y, 22+bw, y+6], fill=(48,48,60))
            drw.rectangle([22, y, 22+int(bw*ratio), y+6], fill=col)
        y += 22

    # 하단 피드백
    drw.line([(10, panel_h-90),(panel_w-10, panel_h-90)], fill=(60,60,80), width=1)
    fb = feedback_text or _ko('정상 범위', 'OK')
    fb_col = (120,245,150) if any(w in str(fb) for w in ['정상','OK','좋','safe']) else (255,130,130)
    drw.text((14, panel_h-78), _ko('피드백', 'Feedback'), font=fMD, fill=(180,180,190))

    max_ch = max(1, (panel_w - 28) // (15 if FONT_KR_OK else 9))
    lines = textwrap.wrap(str(fb), width=max_ch)[:2]
    if not lines:
        lines = [str(fb)[:max_ch]]
    for i, line in enumerate(lines):
        drw.text((14, panel_h-52 + i*22), line, font=fMD, fill=fb_col)

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


print('시각화 헬퍼 v4 로드 완료')
print(f'  VIDEO_VIEW_MODE = {globals().get("VIDEO_VIEW_MODE", "side")}')
print(f'  한글 폰트 = {KOREAN_FONT_PATH if FONT_KR_OK else "없음 → 영어 fallback"}')
print(f'  발너비 bracket 표시 모드 = {sorted(FOOT_BRACKET_VIEW_MODES)}')
print(f'  몸통기울기 허용오차 = ±{TRUNK_LEAN_TOL_DEG}도')


시각화 헬퍼 v4 로드 완료
  VIDEO_VIEW_MODE = side
  한글 폰트 = C:\Windows\Fonts\malgun.ttf
  발너비 bracket 표시 모드 = ['45', '45deg', '45도', 'diagonal', 'front', 'oblique', '사선', '정면']
  몸통기울기 허용오차 = ±8.0도


In [11]:
# ── Cell 7.5: v5 업그레이드 오버라이드 ─────────────────────────
# 목표:
# 1) '값이 큽니다/작습니다' 대신 사람이 이해하기 쉬운 피드백 문장 사용
# 2) neck_lean처럼 측면영상에서 흔들리는 지표는 핵심 오류판정에서 제외
# 3) 피드백이 프레임마다 깜빡이지 않도록 안정화(stabilizer) 적용
# 4) HUD에 U/E/Δ 의미와 현재 구간(phase)을 표시

import textwrap
from collections import deque

# ── 측면영상에서 너무 불안정한 지표 제외 ───────────────────────
# neck_lean은 얼굴/머리 landmark 흔들림 영향이 커서 발표 영상에서 "목머리각 값이..." 같은
# 어색한 피드백을 만들 가능성이 높다. 필요하면 아래 remove 목록에서 빼면 된다.
if str(globals().get('VIDEO_VIEW_MODE', 'side')).lower() in ['side', '측면']:
    _drop_keys = {'neck_lean', 'foot_width', 'hand_width', 'hand_height_diff', 'shoulder_height_diff'}
    rt.EXERCISE_METRICS[EXERCISE] = [
        k for k in rt.EXERCISE_METRICS.get(EXERCISE, []) if k not in _drop_keys
    ]

# 발표용 핵심 오류 후보. 너무 많은 지표가 동시에 뜨지 않도록 1순위 후보를 제한한다.
V5_PRIORITY_KEYS = ['trunk_lean', 'knee_angle', 'hip_angle', 'foot_flatness', 'head_hip_line']

V5_LABELS = {
    'trunk_lean':     _ko('몸통 기울기', 'Trunk lean'),
    'knee_angle':     _ko('무릎각', 'Knee angle'),
    'hip_angle':      _ko('엉덩이각', 'Hip angle'),
    'foot_flatness':  _ko('발바닥 밀착', 'Foot contact'),
    'head_hip_line':  _ko('머리-골반 정렬', 'Head-hip line'),
}


def _safe_float(x, default=None):
    try:
        x = float(x)
        return x if np.isfinite(x) else default
    except Exception:
        return default


def phase_from_metrics(metrics):
    """무릎각 기반의 간단한 스쿼트 구간 표시."""
    k = _safe_float(metrics.get('knee_angle') if isinstance(metrics, dict) else None)
    if k is None:
        return _ko('구간 인식 중', 'Detecting phase')
    if k >= 150:
        return _ko('준비/상승완료', 'Ready / top')
    if k <= 85:
        return _ko('최저점', 'Bottom')
    if k <= 120:
        return _ko('하강·상승 중간', 'Mid phase')
    return _ko('하강/상승 전환', 'Transition')


def explain_delta_feedback(key, delta, user_metrics=None, expert_metrics=None):
    """기존 '값이 큼/작음' 대신 자세 의미 중심 피드백."""
    d = _safe_float(delta, 0.0)
    ad = abs(d)

    if key == 'trunk_lean':
        return _ko('몸통 기울기 과도' if d > 0 else '몸통 기울기 부족',
                   'Trunk lean too large' if d > 0 else 'Trunk lean too small')
    if key == 'knee_angle':
        # knee_angle이 크면 무릎이 덜 접힌 것, 작으면 더 많이 접힌 것에 가깝다.
        return _ko('무릎이 전문가보다 덜 굽혀짐' if d > 0 else '무릎이 전문가보다 많이 굽혀짐',
                   'Knee bends less than expert' if d > 0 else 'Knee bends more than expert')
    if key == 'hip_angle':
        return _ko('엉덩이 접힘이 전문가와 다름', 'Hip angle differs from expert')
    if key == 'foot_flatness':
        return _ko('발바닥/뒤꿈치 접지 확인', 'Check foot contact')
    if key == 'head_hip_line':
        return _ko('머리-골반 정렬 확인', 'Check head-hip alignment')

    label = V5_LABELS.get(key, rt.DISPLAY_LABELS.get(key, key))
    return _ko(f'{label} 차이 확인', f'Check {label}')

# rt.delta_feedback을 항상 v5 문장으로 덮어쓴다.
rt.delta_feedback = explain_delta_feedback


def choose_v5_issue(metrics_now, ex_metrics, deltas, rule_issue=None):
    """
    현재 프레임에서 가장 설명하기 좋은 오류 하나만 선택한다.
    - 몸통기울기는 별도 zone과 연결되므로 최우선
    - neck_lean 등 불안정 지표는 제외
    - threshold를 넘은 delta만 후보로 사용
    """
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))

    # 1) 몸통 기울기: 시각화 zone과 직접 연결되어 가장 직관적
    if 'trunk_lean' in active:
        trunk_issue = evaluate_trunk_lean_issue(metrics_now, ex_metrics, TRUNK_LEAN_TOL_DEG)
        if trunk_issue:
            trunk_issue['priority'] = 100
            trunk_issue['ratio'] = abs(_safe_float(trunk_issue.get('delta'), 0.0)) / max(TRUNK_LEAN_TOL_DEG, 1e-6)
            return trunk_issue

    # 2) delta 기반 핵심 후보
    candidates = []
    for key in V5_PRIORITY_KEYS:
        if key not in active:
            continue
        if key not in deltas:
            continue
        thr = _safe_float(rt.DELTA_THRESHOLDS.get(key))
        d = _safe_float(deltas.get(key))
        if thr is None or thr <= 0 or d is None:
            continue
        ratio = abs(d) / max(thr, 1e-6)
        if ratio < 1.0:
            continue

        # standing/top 구간에서는 head/foot 지표가 튀기 쉬워서 우선순위를 낮춘다.
        k = _safe_float(metrics_now.get('knee_angle'))
        top_phase = k is not None and k >= 150
        base_priority = {
            'knee_angle': 80,
            'hip_angle': 70,
            'foot_flatness': 58 if not top_phase else 35,
            'head_hip_line': 45 if not top_phase else 25,
            'trunk_lean': 100,
        }.get(key, 40)

        candidates.append({
            'key': key,
            'message': explain_delta_feedback(key, d, metrics_now, ex_metrics),
            'landmarks': rt.DELTA_TO_LM.get(key, []),
            'delta': d,
            'ratio': ratio,
            'priority': base_priority + min(ratio, 2.0) * 5,
        })

    if candidates:
        return sorted(candidates, key=lambda x: x.get('priority', 0), reverse=True)[0]

    # 3) 기존 rule 기반 issue가 있으면 fallback
    if rule_issue:
        out = dict(rule_issue)
        out.setdefault('priority', 30)
        out.setdefault('ratio', 1.0)
        return out

    return None


class FeedbackStabilizer:
    """같은 오류가 몇 프레임 연속 나올 때만 피드백을 바꿔 깜빡임을 줄인다."""
    def __init__(self, min_frames=3, hold_frames=8):
        self.min_frames = int(min_frames)
        self.hold_frames = int(hold_frames)
        self.candidate_key = None
        self.candidate_msg = None
        self.candidate_count = 0
        self.stable_issue = None
        self.hold_left = 0

    def update(self, issue):
        if issue is None:
            self.candidate_key = None
            self.candidate_msg = None
            self.candidate_count = 0
            if self.stable_issue is not None and self.hold_left > 0:
                self.hold_left -= 1
                return self.stable_issue
            self.stable_issue = None
            return None

        key = issue.get('key')
        msg = issue.get('message')
        same = (key == self.candidate_key and msg == self.candidate_msg)
        if same:
            self.candidate_count += 1
        else:
            self.candidate_key = key
            self.candidate_msg = msg
            self.candidate_count = 1

        if self.stable_issue is None or self.candidate_count >= self.min_frames:
            self.stable_issue = issue
            self.hold_left = self.hold_frames
        return self.stable_issue


def draw_feedback_banner(frame, feedback_text, issue_key=None,
                         user_metrics=None, expert_metrics=None, phase_text=None):
    """상단 배너를 한 줄 요약 + 현재 구간으로 표시."""
    fb = feedback_text or _ko('정상 범위', 'OK')
    phase = phase_text or phase_from_metrics(user_metrics or {})
    is_ok = any(w in str(fb) for w in ['정상', 'OK', '좋', 'safe'])
    fg = (120, 245, 150) if is_ok else (255, 130, 130)
    line1 = f'{phase}  |  {fb}'
    return draw_text_box_bgr(
        frame, line1, xy=(16, 52), font_size=21,
        fg=fg, bg=(18, 18, 28), alpha=0.78
    )


def draw_issue_markers(frame, lms, deltas, max_items=2):
    """핵심 오류 후보 관절만 표시. 빨간 원이 너무 많이 생기는 것을 방지."""
    H, W = frame.shape[:2]
    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
    allowed = [k for k in V5_PRIORITY_KEYS if k in active]

    items = []
    for k in allowed:
        if k not in deltas:
            continue
        thr = _safe_float(rt.DELTA_THRESHOLDS.get(k))
        d = _safe_float(deltas.get(k))
        if thr is None or thr <= 0 or d is None:
            continue
        ratio = abs(d) / max(thr, 1e-6)
        if ratio >= 1.0:
            items.append((ratio, k, d))
    items = sorted(items, reverse=True)[:max_items]

    for ratio, key, delta in items:
        for idx in rt.DELTA_TO_LM.get(key, []):
            p = lms[idx]
            v = float(getattr(p, 'visibility', 1.0))
            if v < VIS_THR:
                continue
            cx = int(p.x * W)
            cy = int(p.y * H)
            cv2.circle(frame, (cx, cy), 14, C_ERR_CIR, 2, cv2.LINE_AA)
            cv2.circle(frame, (cx, cy),  4, C_ERR_CIR, -1, cv2.LINE_AA)
    return frame


def make_hud_panel(panel_w, panel_h, user_metrics, deltas, count,
                   feedback_text, issue_key, fps, ex_idx, ex_total,
                   exercise='squat', expert_metrics=None, phase_text=None):
    """v5 HUD: U/E/Δ 의미, 구간, 핵심 지표만 크게 표시."""
    img  = PilImage.new('RGB', (panel_w, panel_h), (18, 18, 28))
    drw  = ImageDraw.Draw(img)

    fTITLE = _fnt(28)
    fBIG   = _fnt(23)
    fMD    = _fnt(16)
    fSM    = _fnt(13)

    y = 14
    EX = {'squat':'SQUAT','deadlift':'DEADLIFT','bench':'BENCH PRESS'}

    # 제목/횟수
    drw.text((14, y), EX.get(exercise, exercise.upper()), font=fTITLE, fill=(240, 205, 80))
    drw.text((panel_w-72, y+2), str(count), font=fTITLE, fill=(80, 220, 255))
    drw.text((panel_w-118, y+12), _ko('횟수', 'rep'), font=fSM, fill=(165,165,175))
    y += 44

    # 구간 + 용어 설명
    phase = phase_text or phase_from_metrics(user_metrics or {})
    drw.rounded_rectangle([10, y, panel_w-10, y+38], radius=10, fill=(31,31,45), outline=(65,65,85), width=1)
    drw.text((22, y+9), phase, font=fMD, fill=(230,230,235))
    y += 48
    drw.text((14, y), _ko('U=사용자  E=전문가  Δ=차이', 'U=user  E=expert  Δ=difference'),
             font=fSM, fill=(145,145,155))
    y += 25

    # 핵심 피드백 카드
    fb = feedback_text or _ko('정상 범위', 'OK')
    ok = any(w in str(fb) for w in ['정상', 'OK', '좋', 'safe'])
    card_col = (95, 220, 120) if ok else (255, 115, 115)
    drw.rounded_rectangle([10, y, panel_w-10, y+70], radius=13,
                          fill=(34,34,50), outline=card_col, width=2)
    drw.text((22, y+9), _ko('현재 핵심 피드백', 'Main feedback'), font=fSM, fill=(180,180,190))
    # 너무 긴 문장 줄바꿈
    lines = textwrap.wrap(str(fb), width=max(8, (panel_w-44)//15))[:2]
    if not lines:
        lines = [str(fb)]
    for i, line in enumerate(lines):
        drw.text((22, y+31+i*21), line, font=fMD, fill=card_col)
    y += 84

    # 진행 정보
    prog = _ko(f'전문가 프레임 {ex_idx+1}/{ex_total}   FPS {fps:.1f}',
               f'Expert {ex_idx+1}/{ex_total}   FPS {fps:.1f}')
    drw.text((14, y), prog, font=fSM, fill=(115,115,125))
    y += 24
    drw.line([(10,y),(panel_w-10,y)], fill=(60,60,80), width=1)
    y += 14

    # 지표 목록: 발표용으로 5개만
    metric_list = [
        ('무릎각',        'Knee angle',      'knee_angle',     'deg'),
        ('엉덩이각',      'Hip angle',       'hip_angle',      'deg'),
        ('몸통기울기',    'Trunk lean',      'trunk_lean',     'deg'),
        ('머리-골반정렬', 'Head-hip line',   'head_hip_line',  ''),
        ('발바닥밀착',    'Foot contact',    'foot_flatness',  ''),
    ]

    active = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
    for kr_label, en_label, key, unit in metric_list:
        if key not in active and key not in ['knee_angle', 'hip_angle', 'trunk_lean']:
            continue
        if y > panel_h - 80:
            break

        label = _ko(kr_label, en_label)
        val = user_metrics.get(key) if isinstance(user_metrics, dict) else None
        ex_val = None
        if isinstance(expert_metrics, dict):
            ex_val = expert_metrics.get(key)
        if ex_val is None and isinstance(deltas, dict) and key in deltas and val is not None:
            ex_val = val - deltas.get(key, 0)

        delta = None
        if val is not None and ex_val is not None:
            try:
                delta = float(val) - float(ex_val)
            except Exception:
                delta = None
        elif isinstance(deltas, dict):
            delta = deltas.get(key)

        thr = rt.DELTA_THRESHOLDS.get(key)
        bad = False
        if delta is not None and thr is not None:
            bad = abs(float(delta)) > float(thr)
        col = (255,130,130) if bad else (120,245,150)

        drw.text((14, y), label, font=fMD, fill=(225,225,230))
        drw.text((panel_w-74, y), _ko('주의', 'WARN') if bad else 'OK', font=fMD, fill=col)
        y += 21

        val_line = f"U {_fmt_val(val, unit)} / E {_fmt_val(ex_val, unit)}"
        if delta is not None:
            val_line += f" / Δ {_fmt_val(delta, unit)}"
        drw.text((22, y), val_line, font=fSM, fill=(172,172,182))
        y += 19

        if delta is not None and thr is not None and float(thr) > 0:
            bw = panel_w - 44
            ratio = min(abs(float(delta)) / float(thr), 1.5) / 1.5
            drw.rectangle([22, y, 22+bw, y+5], fill=(48,48,60))
            drw.rectangle([22, y, 22+int(bw*ratio), y+5], fill=col)
        y += 18

    # 하단 설명
    drw.line([(10, panel_h-64),(panel_w-10, panel_h-64)], fill=(60,60,80), width=1)
    note = _ko('빨간 원 = 현재 우선 확인 관절', 'Red circle = joint to check first')
    drw.text((14, panel_h-48), note, font=fSM, fill=(165,165,175))
    note2 = _ko('측면영상: 발너비/양손거리 판정 제외', 'Side view: width metrics excluded')
    drw.text((14, panel_h-28), note2, font=fSM, fill=(115,115,125))

    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

print('v5 피드백 업그레이드 로드 완료')
print(f'  핵심 오류 후보: {V5_PRIORITY_KEYS}')
print(f'  현재 활성 지표: {rt.EXERCISE_METRICS.get(EXERCISE)}')


v5 피드백 업그레이드 로드 완료
  핵심 오류 후보: ['trunk_lean', 'knee_angle', 'hip_angle', 'foot_flatness', 'head_hip_line']
  현재 활성 지표: ['knee_angle', 'hip_angle', 'spine_lean', 'trunk_lean', 'hip_knee_angle', 'foot_flatness', 'head_hip_line']


In [12]:
# ── Cell 8A: MediaPipe PoseLandmarker 모델 로드 ─────────────────
rt.ensure_model(MODEL_PATH)

print('[PoseLandmarker] 로딩 중...')
landmarker = rt.build_landmarker(MODEL_PATH)
print('[PoseLandmarker] 완료')


[모델] pose_landmarker_lite.task 확인 완료
[PoseLandmarker] 로딩 중...
[PoseLandmarker] 완료


In [13]:
# ── Cell 8B: 전문가 JSON 로드 ─────────────────────────────────
expert = rt.load_expert(EXERCISE, EXPERT_JSON_PATH)

ex_total       = expert['total_frames']
ex_frames_data = expert['frames']
print(f'[전문가] {ex_total}프레임, {expert.get("fps",24):.1f}fps')


[전문가] expert_dead2.json 로드 완료  (502프레임, 24.0fps)
[전문가] 502프레임, 24.0fps


In [14]:
# ── Cell 8C: 입력 영상 정보 확인 + 출력 레이아웃 계산 ─────────────
cap_probe = cv2.VideoCapture(USER_VIDEO_PATH)
if not cap_probe.isOpened():
    raise RuntimeError(f'영상 열기 실패: {USER_VIDEO_PATH}')

fps = cap_probe.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

W = int(cap_probe.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap_probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_probe.release()

print(f'[영상] {W}x{H}, {fps:.2f}fps')

# 출력 레이아웃
USER_W = W
USER_H = H

# 전문가 skeleton 영역 폭. 너무 작으면 skeleton이 답답하고, 너무 크면 출력이 과도하게 넓어짐.
EXPERT_W = min(520, max(420, int(W * 0.40)))
PW       = 410   # v5 HUD 카드/설명 공간 확보

OUT_W = USER_W + EXPERT_W + PW
OUT_H = H

print(f'[레이아웃] USER:{USER_W}x{USER_H} | EXPERT:{EXPERT_W}x{OUT_H} | HUD:{PW}x{OUT_H}')
print(f'[출력 크기] {OUT_W}x{OUT_H}')
print(f'[출력 경로] {OUTPUT_PATH}')


[영상] 1080x1920, 29.98fps
[레이아웃] USER:1080x1920 | EXPERT:432x1920 | HUD:410x1920
[출력 크기] 1922x1920
[출력 경로] C:\Users\kojy1\PycharmProjects\capstone\output\dead_v5_trunk_foot_upgrade_model_infer.mp4


In [15]:
# ── Cell 9A: rt 호환 fallback 객체 준비 ───────────────────────
# rt 모듈에 없는 객체는 notebook이 멈추지 않도록 fallback 생성.

import numpy as np

if not hasattr(rt, 'WINDOW_SIZE'):
    rt.WINDOW_SIZE = 5

if not hasattr(rt, 'PoseLandmarksConns'):
    rt.PoseLandmarksConns = SKEL_CONNS


# MetricsBuffer 없을 경우 fallback
if not hasattr(rt, 'MetricsBuffer'):
    class MetricsBuffer:
        def __init__(self, window=5):
            self.window = int(window)
            self.buf = []

        def push(self, metrics):
            self.buf.append(dict(metrics))
            if len(self.buf) > self.window:
                self.buf.pop(0)

        def get_avg(self):
            if not self.buf:
                return {}

            keys = set()
            for m in self.buf:
                keys.update(m.keys())

            out = {}
            for k in keys:
                vals = []
                for m in self.buf:
                    v = m.get(k)
                    if isinstance(v, (int, float, np.floating)) and np.isfinite(v):
                        vals.append(float(v))
                if vals:
                    out[k] = float(np.mean(vals))
            return out

    rt.MetricsBuffer = MetricsBuffer
    print('보정: rt.MetricsBuffer 임시 생성 완료')


# RuleState 없을 경우 fallback
if not hasattr(rt, 'RuleState'):
    class RuleState:
        def __init__(self, exercise='squat'):
            self.exercise = exercise

        def evaluate(self, metrics):
            # 세부 rule이 없으면 delta 기반 비교만 사용
            return []

    rt.RuleState = RuleState
    print('보정: rt.RuleState 임시 생성 완료')


# SideViewRepCounter 없을 경우 fallback
if not hasattr(rt, 'SideViewRepCounter'):
    class SideViewRepCounter:
        def __init__(self, exercise='squat'):
            self.exercise = exercise
            self.count = 0
            self.was_down = False

        def update(self, metrics):
            if self.exercise == 'squat':
                angle = metrics.get('knee_angle')
                if angle is None:
                    return False

                # squat 기준 간단 카운터
                # 내려감: 무릎각 작아짐 / 올라옴: 무릎각 커짐
                low_thr  = 105
                high_thr = 150

                if angle < low_thr:
                    self.was_down = True

                if self.was_down and angle > high_thr:
                    self.count += 1
                    self.was_down = False
                    return True

            return False

    rt.SideViewRepCounter = SideViewRepCounter
    print('보정: rt.SideViewRepCounter 임시 생성 완료')


if not hasattr(rt, 'pick_worst_issue'):
    def pick_worst_issue(issues):
        if not issues:
            return None
        return issues[0]

    rt.pick_worst_issue = pick_worst_issue
    print('보정: rt.pick_worst_issue 임시 생성 완료')


if not hasattr(rt, 'compute_deltas'):
    def compute_deltas(user_metrics, expert_metrics, exercise='squat'):
        deltas = {}
        keys = rt.EXERCISE_METRICS.get(exercise, list(user_metrics.keys()))

        for k in keys:
            if k not in user_metrics or k not in expert_metrics:
                continue

            uv = user_metrics.get(k)
            ev = expert_metrics.get(k)

            if uv is None or ev is None:
                continue

            try:
                deltas[k] = float(uv) - float(ev)
            except Exception:
                pass

        return deltas

    rt.compute_deltas = compute_deltas
    print('보정: rt.compute_deltas 임시 생성 완료')


if not hasattr(rt, 'select_worst_delta'):
    def select_worst_delta(deltas):
        if not deltas:
            return None

        worst_key = None
        worst_delta = None
        worst_ratio = 0.0

        for k, d in deltas.items():
            thr = rt.DELTA_THRESHOLDS.get(k)
            if thr is None or thr <= 0:
                continue

            ratio = abs(float(d)) / max(float(thr), 1e-6)
            if ratio > worst_ratio:
                worst_ratio = ratio
                worst_key = k
                worst_delta = d

        if worst_key is None or worst_ratio < 1.0:
            return None

        return worst_key, worst_delta

    rt.select_worst_delta = select_worst_delta
    print('보정: rt.select_worst_delta 임시 생성 완료')


if not hasattr(rt, 'delta_feedback'):
    def delta_feedback(key, delta):
        label = rt.DISPLAY_LABELS.get(key, key)

        if delta > 0:
            return f'{label} 값이 전문가보다 큽니다'
        return f'{label} 값이 전문가보다 작습니다'

    rt.delta_feedback = delta_feedback
    print('보정: rt.delta_feedback 임시 생성 완료')


보정: rt.MetricsBuffer 임시 생성 완료
보정: rt.RuleState 임시 생성 완료
보정: rt.SideViewRepCounter 임시 생성 완료
보정: rt.pick_worst_issue 임시 생성 완료
보정: rt.compute_deltas 임시 생성 완료
보정: rt.select_worst_delta 임시 생성 완료


In [16]:
# ── Cell 9B: ST-GCN 모델 helper 정의 ───────────────────────────
# ── ST-GCN 모델 기반 종목 인식 / 횟수 카운터 ─────────────────────
MODEL_CLASS_LABELS = {
    'squat': 'SQUAT',
    'benchpress': 'BENCH PRESS',
    'deadlift': 'DEADLIFT',
}


def resolve_project_path(path):
    if path is None:
        return None
    text = os.fspath(path)
    if os.path.isabs(text):
        return text
    return os.path.join(BASE, text)


class ModelRepCounter:
    """rt.SideViewRepCounter 호환 wrapper.

    - observe_landmarks(lms): ST-GCN action/phase/count 갱신
    - update(metrics): 기존 메인 루프의 update(metrics) 형태를 유지하면서
      모델의 last_increment만 반환
    """

    def __init__(self, infer):
        self.infer = infer
        self.count = 0
        self.was_down = False
        self.last_increment = False
        self.state = getattr(infer, 'state', None)

    def reset(self):
        self.count = 0
        self.was_down = False
        self.last_increment = False
        self.infer.reset()
        self.state = self.infer.clear_frame()

    def clear_frame(self):
        self.last_increment = False
        self.state = self.infer.clear_frame()
        self.count = self.state.count
        self.was_down = (self.state.phase == 'down')
        return self.state

    def observe_landmarks(self, lms):
        self.state = self.infer.update(lms)
        self.count = self.state.count
        self.was_down = (self.state.phase == 'down')
        self.last_increment = self.state.last_increment
        return self.state

    def update(self, metrics):
        return self.last_increment


class ExerciseSmoother:
    """ST-GCN action prediction을 HUD 표시용으로 안정화."""

    def __init__(self, window=7, fallback='squat'):
        from collections import deque
        self.window = deque(maxlen=max(1, int(window)))
        self.current = fallback

    def reset(self, fallback='squat'):
        self.window.clear()
        self.current = fallback

    def update(self, state, fallback='squat'):
        if not MODEL_DRIVEN_RECOGNITION:
            self.current = fallback
            return fallback
        if state is None or not getattr(state, 'ready', False):
            return self.current or fallback
        pred = getattr(state, 'pred_class', None)
        conf = float(getattr(state, 'action_confidence', 0.0) or 0.0)
        if pred not in CLASS_LIST or conf < MODEL_ACTION_CONFIDENCE_MIN:
            return self.current or fallback
        self.window.append(pred)
        counts = {}
        latest_index = {}
        for i, value in enumerate(self.window):
            counts[value] = counts.get(value, 0) + 1
            latest_index[value] = i
        self.current = max(counts, key=lambda value: (counts[value], latest_index[value]))
        return self.current


def init_rep_counter_with_model(exercise):
    """모델 카운터를 우선 생성하고, 설정상 허용될 때만 규칙 기반 fallback."""
    if not MODEL_DRIVEN_COUNT:
        print('[AI] MODEL_DRIVEN_COUNT=False → 기존 규칙 기반 counter 사용')
        return rt.SideViewRepCounter(exercise), None, 'AI count disabled; rule counter active'

    try:
        infer = RealtimeSTGCNInfer(
            checkpoint_path=resolve_project_path(TRAINED_MODEL_CHECKPOINT),
            device=TRAINED_MODEL_DEVICE,
            clip_len_override=TRAINED_MODEL_CLIP_LEN_OVERRIDE,
            smooth_window=TRAINED_MODEL_SMOOTH_WINDOW,
            min_up_len=TRAINED_MODEL_MIN_UP_LEN,
            inference_interval=TRAINED_MODEL_INFERENCE_INTERVAL,
            required=TRAINED_MODEL_REQUIRED,
        )
    except Exception as e:
        if TRAINED_MODEL_REQUIRED:
            raise RuntimeError(f'ST-GCN 모델 초기화 실패: {e}') from e
        print(f'[AI] ST-GCN 모델 초기화 실패 → 규칙 기반 fallback: {e}')
        return rt.SideViewRepCounter(exercise), None, f'AI unavailable: {e}'

    if not infer.available:
        msg = infer.unavailable_reason or 'unknown reason'
        if TRAINED_MODEL_REQUIRED:
            raise RuntimeError(f'ST-GCN 모델 사용 불가: {msg}')
        print(f'[AI] ST-GCN 모델 사용 불가 → 규칙 기반 fallback: {msg}')
        return rt.SideViewRepCounter(exercise), infer.state, f'AI unavailable: {msg}'

    status = f'ST-GCN loaded: {infer.checkpoint_path} | clip_len={infer.clip_len}'
    if infer.warning:
        status += f' | {infer.warning}'
    print(f'[AI] {status}')
    return ModelRepCounter(infer), infer.state, status


def make_model_phase_text(state):
    if state is None:
        return _ko('AI 구간 인식 대기', 'AI phase pending')
    if not getattr(state, 'available', False):
        return f"AI unavailable: {getattr(state, 'unavailable_reason', '')}"
    if not getattr(state, 'ready', False):
        return _ko('AI 워밍업 중', 'AI warming up')
    pred = getattr(state, 'pred_class', None) or '?'
    aconf = float(getattr(state, 'action_confidence', 0.0) or 0.0)
    phase = getattr(state, 'phase', '?')
    pconf = float(getattr(state, 'phase_confidence', 0.0) or 0.0)
    return f'AI {pred} {aconf:.2f} | phase {phase} {pconf:.2f}'


In [17]:
# ── Cell 9C: 버퍼/규칙/모델 카운터 상태 객체 생성 ───────────────
# 상태 객체 생성
buf         = rt.MetricsBuffer(window=rt.WINDOW_SIZE)
rules       = rt.RuleState(EXERCISE)
rep_counter, model_state, model_status = init_rep_counter_with_model(EXERCISE)
exercise_smoother = ExerciseSmoother(MODEL_ACTION_SMOOTH_WINDOW, fallback=EXERCISE)
display_exercise = EXERCISE

renderer = fo.FeedbackRenderer(fps=fps)


[AI] ST-GCN loaded: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\temporal_head_full_ablation\mlp_h128_c16_pw_2.0_ts2_do03_aug1\best_ep013.pt | clip_len=16


In [18]:
# ── Cell 9D: feedback overlay renderer 설정 ───────────────────
# ── 기존 시각화 레이어 복구 ─────────────────────────────────────
# Claude 코드에서 phase/tempo만 켜져 있어서 무릎각 arc, metric bar, acl, ktg 등이 사라졌음.
# feedback_overlay.py 버전에 따라 없는 레이어는 자동 skip.
# 기본적으로 예전 노트북에서 쓰던 검증된 레이어만 켠다.
# bar_proxy / heatmap / trail 같은 이름은 feedback_overlay.py 버전에 따라 없을 수 있으므로
# renderer 내부 registry에서 확인될 때만 켠다.
# metricbar는 한글/숫자가 작게 겹쳐 깨져 보이므로 오른쪽 HUD로 대체.
# ktg도 버전에 따라 작은 텍스트가 겹치면 아래 리스트에서 빼도 된다.
CORE_LAYERS = ['phase', 'tempo', 'depth', 'arc', 'acl', 'ktg']
OPTIONAL_LAYERS = ['bar_proxy', 'heatmap', 'trail']

available_layers = None
for attr in ['layers', 'layer_map', 'registry', 'renderers', 'available_layers']:
    obj = getattr(renderer, attr, None)
    if isinstance(obj, dict):
        available_layers = set(obj.keys())
        break
    if isinstance(obj, (set, list, tuple)):
        available_layers = set(obj)
        break

if available_layers is None:
    VISIBLE_LAYERS = CORE_LAYERS
else:
    VISIBLE_LAYERS = [x for x in CORE_LAYERS + OPTIONAL_LAYERS if x in available_layers]

enabled_layers = []
skipped_layers = []

for layer in VISIBLE_LAYERS:
    try:
        renderer.enable(layer)
        enabled_layers.append(layer)
    except Exception as e:
        skipped_layers.append((layer, str(e)))

# depth 목표선 설정
try:
    expert_target_y_norm = fo.extract_expert_depth_target(ex_frames_data, EXERCISE)
    renderer.depth_line.set_expert_target(expert_target_y_norm, OUT_H)
except Exception as e:
    print(f'[참고] depth 목표선 설정 skip: {e}')

print('상태 객체 / 렌더러 설정 완료')
print(f'  피드백 기준 종목: {EXERCISE} | MetricsBuffer 윈도우: {rt.WINDOW_SIZE}')
print(f'  모델 상태: {model_status}')
print(f'  ON 레이어: {enabled_layers}')
if skipped_layers:
    print(f'  SKIP 레이어: {[x[0] for x in skipped_layers]}')


상태 객체 / 렌더러 설정 완료
  피드백 기준 종목: 모델 작동안하는중 | MetricsBuffer 윈도우: 5
  모델 상태: ST-GCN loaded: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\temporal_head_full_ablation\mlp_h128_c16_pw_2.0_ts2_do03_aug1\best_ep013.pt | clip_len=16
  ON 레이어: ['phase', 'tempo', 'depth', 'arc', 'acl', 'ktg']


In [19]:
# ── Cell 10: 메인 처리 루프 v5 ───────────────────────────────────
# v5 핵심:
# - 오류 후보를 1개로 안정화해서 피드백 깜빡임 감소
# - '값이 큼/작음' 대신 의미 있는 문장으로 표시
# - HUD에 현재 구간, U/E/Δ 의미, 핵심 피드백을 크게 표시

# Cell 10만 다시 실행해도 되도록 cap/out을 여기서 새로 연다.
cap = cv2.VideoCapture(USER_VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f'영상 열기 실패: {USER_VIDEO_PATH}')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (OUT_W, OUT_H))

if not out.isOpened():
    raise RuntimeError(f'VideoWriter 열기 실패: {OUTPUT_PATH}')

frame_idx = 0
t_start   = time.perf_counter()
feedback_stabilizer = FeedbackStabilizer(min_frames=3, hold_frames=8)
# Cell 10만 재실행해도 HUD 표시 종목은 현재 피드백 기준으로 초기화
display_exercise = EXERCISE

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if rt.MIRROR_CAMERA:
        frame = cv2.flip(frame, 1)

    # 원본 비율 유지
    user_canvas = frame.copy()

    elapsed_ms = int(frame_idx * 1000 / fps)

    # 전문가 프레임 인덱스 & 지표
    ex_idx     = rt.get_expert_frame_idx(elapsed_ms, expert)
    ex_pack    = ex_frames_data[ex_idx]
    ex_lms_raw = ex_pack['landmarks']
    ex_metrics = compute_expert_metrics_plus(
        ex_lms_raw,
        ex_pack.get('metrics', {}),
        EXERCISE
    )

    # 사용자 landmark 추출: 원본 frame 기준
    lms = rt.extract_landmarks(landmarker, user_canvas, elapsed_ms)

    feedback    = ''
    issue_key   = None
    bad_indices = set()
    deltas      = {}
    user_avg    = {}
    raw         = {}
    metrics_now = {}
    phase_text  = _ko('구간 인식 중', 'Detecting phase')

    if lms:
        try:
            raw = compute_user_metrics_plus(lms, EXERCISE)
            buf.push(raw)
            user_avg = buf.get_avg()
        except Exception as e:
            feedback = f'지표 계산 오류: {e}'
            print('[지표 계산 오류]', e)
            user_avg = {}

        metrics_now = user_avg if user_avg else raw
        phase_text  = phase_from_metrics(metrics_now)

        # ST-GCN 종목/phase/count 갱신.
        # 규칙 기반 각도 counter가 아니라 모델 phase transition으로 rep_counter.count가 증가합니다.
        if isinstance(rep_counter, ModelRepCounter):
            model_state = rep_counter.observe_landmarks(lms)
            display_exercise = exercise_smoother.update(model_state, EXERCISE)
            phase_text = make_model_phase_text(model_state)
        else:
            display_exercise = EXERCISE

        # 횟수 카운트: ModelRepCounter면 metrics 계산 성공 여부와 무관하게
        # 방금 observe_landmarks(lms)에서 갱신된 모델 phase/count를 사용합니다.
        count_increment = False
        if isinstance(rep_counter, ModelRepCounter):
            count_increment = rep_counter.update(metrics_now)
        elif metrics_now:
            count_increment = rep_counter.update(metrics_now)

        if count_increment:
            _label = getattr(model_state, 'pred_class', None) if isinstance(rep_counter, ModelRepCounter) else display_exercise
            print(f'[{(_label or display_exercise).upper()}] {rep_counter.count}회')

        if metrics_now:
            # 전문가 delta 계산 후 현재 view에서 실제 사용하는 지표만 유지
            deltas = rt.compute_deltas(metrics_now, ex_metrics, EXERCISE)
            active_keys = set(rt.EXERCISE_METRICS.get(EXERCISE, []))
            deltas = {k: v for k, v in deltas.items() if k in active_keys}

            # 기존 rule 기반 issue는 fallback으로만 사용
            rule_issues = rules.evaluate(metrics_now)
            rule_issue  = rt.pick_worst_issue(rule_issues)

            # v5: 가장 설명하기 좋은 issue 1개 선택 후 안정화
            candidate_issue = choose_v5_issue(metrics_now, ex_metrics, deltas, rule_issue)
            stable_issue = feedback_stabilizer.update(candidate_issue)

            if stable_issue:
                issue_key   = stable_issue.get('key')
                feedback    = stable_issue.get('message', '')
                bad_indices = set(stable_issue.get('landmarks', []))
            else:
                feedback = _ko('정상 범위', 'OK')

        # 기존 feedback_overlay 렌더
        try:
            user_canvas = renderer.render(
                frame            = user_canvas,
                user_metrics_raw = metrics_now,
                expert_metrics   = ex_metrics,
                deltas           = deltas,
                lms              = lms,
                connections      = rt.PoseLandmarksConns,
                metric_to_lm     = rt.DELTA_TO_LM,
                exercise         = EXERCISE
            )
        except Exception as e:
            # renderer 하나가 실패해도 전체 영상 생성은 계속 진행
            print(f'[renderer 오류] frame={frame_idx}: {e}')

        # 추가 레이어 1: 발너비 bracket
        # 측면이면 VIDEO_VIEW_MODE='side'라서 자동으로 숨김.
        user_canvas = draw_foot_bracket(
            user_canvas, lms, metrics_now, view_mode=VIDEO_VIEW_MODE
        )

        # 추가 레이어 2: 몸통기울기 허용범위 zone
        user_canvas = draw_trunk_lean_zone(
            user_canvas, lms,
            metrics_now,
            ex_metrics,
            bad_indices,
            tol_deg=TRUNK_LEAN_TOL_DEG
        )

        # 추가 레이어 3: threshold 초과 핵심 관절 원 표시
        user_canvas = draw_issue_markers(
            user_canvas, lms, deltas, max_items=2
        )

        # 추가 레이어 4: 큰 피드백 배너
        user_canvas = draw_feedback_banner(
            user_canvas, feedback, issue_key, metrics_now, ex_metrics, phase_text=phase_text
        )

    else:
        if isinstance(rep_counter, ModelRepCounter):
            model_state = rep_counter.clear_frame()
            phase_text = make_model_phase_text(model_state)
            display_exercise = exercise_smoother.current or EXERCISE
        feedback = _ko('자세를 인식할 수 없습니다. 전신이 보이게 해주세요.',
                       'Pose not detected. Show full body.')

    # 사용자 레이블
    cv2.putText(
        user_canvas, 'USER',
        (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX, 0.75,
        (60, 210, 255), 2, cv2.LINE_AA
    )

    # 전문가 skeleton canvas
    expert_canvas = draw_expert_on_canvas(
        EXPERT_W, OUT_H, ex_lms_raw, bad_indices
    )

    # HUD
    elapsed = time.perf_counter() - t_start
    cur_fps = (frame_idx + 1) / max(elapsed, 1e-6)

    hud_panel = make_hud_panel(
        panel_w        = PW,
        panel_h        = OUT_H,
        user_metrics   = user_avg if user_avg else raw,
        deltas         = deltas,
        count          = rep_counter.count,
        feedback_text  = feedback,
        issue_key      = issue_key,
        fps            = cur_fps,
        ex_idx         = ex_idx,
        ex_total       = ex_total,
        # HUD 제목은 모델 종목 인식값을 smoothing한 display_exercise로 표시합니다.
        # deltas/피드백/전문가 비교 기준은 기존 EXERCISE와 expert JSON을 유지합니다.
        exercise       = display_exercise,
        expert_metrics = ex_metrics,
        phase_text     = phase_text
    )

    # 최종 합성: [원본 USER | EXPERT | HUD]
    final_frame = np.hstack([user_canvas, expert_canvas, hud_panel])

    # 영역 구분선
    cv2.line(final_frame, (USER_W, 0), (USER_W, OUT_H), (50, 50, 70), 2)
    cv2.line(final_frame, (USER_W + EXPERT_W, 0), (USER_W + EXPERT_W, OUT_H), (50, 50, 70), 2)

    out.write(final_frame)
    frame_idx += 1

    if frame_idx % 30 == 0:
        ai = make_model_phase_text(model_state) if isinstance(rep_counter, ModelRepCounter) else 'rule-counter'
        print(f'[처리 중] {frame_idx}f | rep={rep_counter.count} | display={display_exercise} | {ai} | feedback={feedback} | fps={cur_fps:.1f}')

cap.release()
out.release()

try:
    landmarker.close()
except Exception:
    pass

print(f'\n완료! 총 {frame_idx} frames')
print(f'총 횟수: {rep_counter.count}회')
if isinstance(rep_counter, ModelRepCounter):
    print(f'최종 모델 상태: {make_model_phase_text(model_state)}')
print(f'출력: {OUTPUT_PATH}')


C:\Users\kojy1\anaconda3\envs\capstone\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[처리 중] 30f | rep=0 | display=deadlift | AI deadlift 1.00 | phase down 0.53 | feedback=몸통 기울기 과도 | fps=8.8
[처리 중] 60f | rep=0 | display=deadlift | AI deadlift 1.00 | phase down 0.54 | feedback=몸통 기울기 과도 | fps=8.7
[처리 중] 90f | rep=0 | display=deadlift | AI deadlift 1.00 | phase down 0.52 | feedback=몸통 기울기 과도 | fps=9.0
[처리 중] 120f | rep=0 | display=deadlift | AI deadlift 1.00 | phase down 0.60 | feedback=몸통 기울기 과도 | fps=8.5
[처리 중] 150f | rep=0 | display=deadlift | AI deadlift 0.59 | phase ready 0.64 | feedback=몸통 기울기 과도 | fps=8.4
[처리 중] 180f | rep=0 | display=deadlift | AI deadlift 0.98 | phase up 0.49 | feedback=몸통 기울기 부족 | fps=8.4
[DEADLIFT] 1회
[DEADLIFT] 2회
[처리 중] 210f | rep=2 | display=deadlift | AI deadlift 0.64 | phase ready 0.36 | feedback=몸통 기울기 부족 | fps=8.4
[처리 중] 240f | rep=2 | display=squat | AI 워밍업 중 | feedback=자세를 인식할 수 없습니다. 전신이 보이게 해주세요. | fps=8.3
[처리 중] 270f | rep=2 | display=squat | AI squat 0.55 | phase up 0.71 | feedback=몸통 기울기 부족 | fps=8.4
[SQUAT] 3회
[SQUAT] 4회
[처리 중] 

In [20]:
# ── Cell 11: 저장 확인만 하기 ─────────────────────────
import os

if not os.path.exists(OUTPUT_PATH):
    print('출력 파일 없음 -- Cell 10 정상 실행 여부 확인')
else:
    size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
    print(f'저장 완료!')
    print(f'파일 크기: {size_mb:.1f} MB')
    print(f'저장 위치: {OUTPUT_PATH}')

저장 완료!
파일 크기: 72.2 MB
저장 위치: C:\Users\kojy1\PycharmProjects\capstone\output\dead_v5_trunk_foot_upgrade_model_infer.mp4


In [21]:
# ── Cell 12: 단일 프레임 디버그 (선택 실행) ──────────────────────
# 메인 루프 실행 전에 시각화를 점검할 때 사용.
# Cell 8 (모델/영상 로드) 이후에 실행 가능.

from matplotlib import pyplot as plt


def debug_frame(video_path, frame_no=60, ex_fi=30):
    """
    지정 프레임의 사용자/전문가 시각화를 matplotlib으로 확인.
    발너비 bracket, 몸통기울기 zone, 전문가 skeleton이 제대로 나오는지 점검용.
    """
    cap_d = cv2.VideoCapture(video_path)
    cap_d.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ret, frame = cap_d.read()
    cap_d.release()
    if not ret:
        print(f'프레임 {frame_no} 읽기 실패')
        return

    uc = frame.copy()
    elapsed_ms = int(frame_no * 1000 / fps)
    lms_d = rt.extract_landmarks(landmarker, uc, elapsed_ms)

    ex_fi = max(0, min(int(ex_fi), len(ex_frames_data)-1))
    ex_pack_d = ex_frames_data[ex_fi]
    ex_lms_d  = ex_pack_d['landmarks']
    ex_met_d  = compute_expert_metrics_plus(ex_lms_d, ex_pack_d.get('metrics', {}), EXERCISE)

    raw_d = {}
    if lms_d:
        raw_d = compute_user_metrics_plus(lms_d, EXERCISE)
        trunk_issue = evaluate_trunk_lean_issue(raw_d, ex_met_d, TRUNK_LEAN_TOL_DEG)
        bad_d = set(trunk_issue.get('landmarks', [])) if trunk_issue else set()

        uc = draw_foot_bracket(uc, lms_d, raw_d, view_mode=VIDEO_VIEW_MODE)
        uc = draw_trunk_lean_zone(uc, lms_d, raw_d, ex_met_d, bad_d, tol_deg=TRUNK_LEAN_TOL_DEG)

        print('=== 지표 계산 결과 ===')
        print(f'VIDEO_VIEW_MODE: {VIDEO_VIEW_MODE}')
        print(f'발너비 bracket 표시 여부: {should_draw_foot_bracket(lms_d, VIDEO_VIEW_MODE)}')
        if trunk_issue:
            print(f'몸통기울기 피드백: {trunk_issue["message"]}')
        else:
            print('몸통기울기 피드백: 정상 범위')
        for k, v in sorted(raw_d.items()):
            print(f'  {k:30s}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
    else:
        print('landmark 미검출')

    ec = draw_expert_on_canvas(EXPERT_W, OUT_H, ex_lms_d, set())
    combined = np.hstack([uc, ec])

    plt.figure(figsize=(14, 7))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title(f'프레임 {frame_no} | 좌: USER  우: EXPERT')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# 60번 프레임 / 전문가 30번 프레임 확인
# debug_frame(USER_VIDEO_PATH, frame_no=60, ex_fi=30)


---
## 📌 v5 주요 수정 사항 요약

| 수정 항목 | 기존 문제 | v5 수정 방법 |
|---|---|---|
| 피드백 문장 | `값이 전문가보다 큽니다/작습니다`가 무엇을 의미하는지 직관적이지 않음 | `몸통 기울기 과도`, `무릎이 덜 굽혀짐`, `발바닥/뒤꿈치 접지 확인`처럼 의미 중심 문장으로 변환 |
| 피드백 깜빡임 | 프레임마다 최악 지표가 바뀌면서 문장이 계속 바뀜 | `FeedbackStabilizer(min_frames=3, hold_frames=8)`로 같은 오류가 몇 프레임 유지될 때만 변경 |
| 측면영상 지표 | 발너비, 양손거리, 목머리각이 측면에서 흔들리며 오검출 유발 | 측면영상에서는 `neck_lean`, `foot_width`, `hand_width`, `hand_height_diff`, `shoulder_height_diff`를 핵심판정에서 제외 |
| HUD | 숫자는 있지만 U/E/Δ 의미가 불명확함 | HUD 상단에 `U=사용자, E=전문가, Δ=차이` 설명 추가 |
| 빨간 원 | 여러 관절이 동시에 표시되어 복잡함 | 핵심 오류 후보 2개까지만 표시 |

---
## ⚙️ 조정 포인트

```python
TRUNK_LEAN_TOL_DEG = 8.0      # 몸통기울기 허용오차
FeedbackStabilizer(min_frames=3, hold_frames=8)
```

- 피드백이 너무 늦게 바뀌면 `min_frames=2`로 낮추면 됩니다.
- 피드백이 너무 자주 바뀌면 `min_frames=4`, `hold_frames=10` 정도로 높이면 됩니다.
- 측면영상에서 발너비는 실제 너비 판단이 어려우므로 계속 제외하는 것이 맞습니다.


---
## 🤖 ST-GCN 모델 인식 버전 추가 사항
- `RealtimeSTGCNInfer`를 불러와 action head로 종목을 계속 예측합니다.
- phase head(`ready/down/up`)의 전환으로 횟수를 카운트합니다.
- `TRAINED_MODEL_REQUIRED=True`이면 모델 로드 실패 시 규칙 기반으로 조용히 넘어가지 않고 에러를 냅니다.
- 자세 피드백/전문가 비교 기준은 기존 `EXERCISE`와 expert JSON을 유지하고, HUD 제목/구간 표시는 모델 출력을 사용합니다.


---
## 🧩 셀 구조 정리
- Cell 3A: 경로 입력만 담당합니다.
- Cell 3B: 모델/실행 옵션만 담당합니다.
- Cell 3C: 경로 검증과 alias 설정만 담당합니다.
- Cell 8A/8B/8C: PoseLandmarker 로드, 전문가 JSON 로드, 영상/레이아웃 계산을 분리했습니다.
- Cell 9A~9D: fallback, ST-GCN helper, 상태 객체, renderer 설정을 분리했습니다.


---
## 📦 Import 제거
- 외부 `realtime_compare_side.py` 참조 제거: 필요한 rt 런타임 함수/상수는 Cell 4A에 내장했습니다.
- 외부 `feedback_overlay.py` 참조 제거: feedback overlay 코드는 Cell 4B에 내장했습니다.
- 외부 `model/realtime_stgcn_infer.py` 참조 제거: ST-GCN 모델/추론 코드는 Cell 4C에 내장했습니다.
